[<img src="imagens/colab-badge.png" style="width:16%; vertical-align:middle;">](https://colab.research.google.com/github/fzampirolli/pdi-vc/blob/master/notebooks_alunos/py.en/cap09/cap09.EPs_aluno.ipynb)
[<img src="imagens/github-badge.png" style="width:19%; vertical-align:middle;">](https://github.com/fzampirolli/pdi-vc)

## 💻 **Practical Section with Programming Exercises**

The present list of **Programming Exercises (PEs)** consolidates the theoretical formulations presented throughout Chapter 9 — Deep Learning for Computer Vision — through an applied practical track. Unlike the training of complete neural networks with PyTorch, which requires execution time and, at times, a GPU, the PEs in this chapter isolate the **intermediate quantities** of a real deep learning pipeline — the output of a single convolutional layer, the result of a pooling operation, the count of trainable parameters of an architecture, the overlap between candidate bounding boxes, the quality of a segmentation mask, and the non-maximum suppression filter — allowing each step of the reasoning to be manually validated without relying on machine learning libraries or actual training.

The sequencing of the exercises reproduces the conceptual flow of the chapter and increases in difficulty at each step: it begins with the manual calculation of the output of a **learned convolutional layer** (🟢), based on an already trained kernel and bias; it proceeds to the **pooling** operation (🟢, max and average), which reduces spatial resolution between convolutional blocks; it continues with the **count of trainable parameters** (🟡) of a complete CNN architecture, evidencing why weight sharing makes these networks so much more economical than an equivalent fully connected layer; it delves into the calculation of **Intersection over Union (IoU)** and **Non-Maximum Suppression (NMS)** (🟡), a post-processing step common to detectors such as Faster R-CNN and YOLO; it moves on to the **evaluation of segmentation masks** (🟠) using the same IoU and Dice metrics employed to compare the U-Net with the classic morphological baseline; and it concludes with an **integrated pipeline** (🔴), combining the output of an object detector (after NMS) with a real-world measurement by scale reference — the same principle of photogrammetry studied in the final integration of the chapter.

Whenever it makes sense, each exercise points to methods from the didactic library `morph.py` (the same one used throughout the chapter, imported as `mm`) that solve a step of the problem or serve as a reference to check your own calculations — without, however, replacing the reasoning that you must implement.

> ### ❗ Guidelines for Solving the Programming Exercises
>
> In all exercises of this chapter, the discretization or numerical rounding steps must employ standard rounding to the nearest integer (*round half away from zero*), mitigating ambiguities in values with a fraction exactly equal to $0{,}5$. Unless explicitly stated otherwise: (i) the "convolution" operation follows the convention adopted by deep learning frameworks — **cross-correlation**, without spatial inversion of the kernel, exactly as presented in the "Convolutional Layer" section; (ii) padding is performed with zeros; (iii) bounding boxes are specified in the corner-to-corner format $(x_1, y_1, x_2, y_2)$, with $x_1 < x_2$ and $y_1 < y_2$; and (iv) vectors/matrices follow indexing starting from $0$, with the `[row][column]` convention for two-dimensional structures.

### 🎯 Objective of this Notebook

This notebook allows you to develop, validate, organize, and test solutions for **Programming Exercises (EPs)** in interactive environments, such as Colab, using the same test cases as Moodle, and copying them there only when registering the official grade.

#### *Download*

Download `morph.py` and `testsuite.py` by running the cell below:

In [ ]:
import os, urllib.request

url = "https://raw.githubusercontent.com/fzampirolli/pdi-vc/master/morph/config.py"
if not os.path.exists("config.py"):
    urllib.request.urlretrieve(url, "config.py")

import config
config.setup(testsuite=True)
from morph import mm
from testsuite import TestSuite

#### Running the Tests
To evaluate the tests, run `TestSuite("EP09_01.extension").run()` in a new cell, replacing the extension with that of the language used (`.py`, `.java`, `.c`, `.cpp`, `.js`, or `.r`). The system downloads the test cases from GitHub, runs the program, and calculates the grade automatically.

To test Python code directly, without saving a file, use `run_code(code)` by passing the code as a *string* in a variable named `code`:

```python
code = """
# ... your code here ...
"""
TestSuite("EP09_01").run_code(code)
```

### EP09_01 🟢 Manual 2D Convolution (Forward Pass of a Learned Layer)

PyTorch, presented in this chapter, executes `nn.Conv2d(x)` in a single call — but behind it is merely the cross-correlation between a (already trained) kernel and a neighborhood of the input, followed by the sum of a bias and an activation, exactly as formalized in the "Convolutional Layer" Section. The essential difference compared to the fixed-kernel convolution of Chapter 3 is that, here, the kernel and bias values **already come ready** (as if they had been learned by gradient), and it is up to you to manually reproduce the forward pass that the framework executes internally.

Before training a real CNN, you have been tasked with implementing this forward pass from scratch, for a single convolutional layer with a single input channel and a single output filter, including support for arbitrary padding and stride.

#### 📋 Implementation Guidelines

1. **Input:** Read the dimensions $H \times W$ of the input feature map and, subsequently, its $H \times W$ real values.
   
2. **Kernel and bias:** Read the dimensions $k_h \times k_w$ of the (already trained) kernel, its real values, and the bias $b$ (real, scalar).
   
3. **Hyperparameters:** Read the padding $p$ (integer, number of zeros added to each border) and the stride $s$ (integer, sliding step).
   
4. **Padding:** Add $p$ zeros to each of the four borders of the input map before the correlation.
   
5. **Cross-correlation:** For each output position $(i, j)$, compute
   $$
   z(i,j) = b + \sum_{u=0}^{k_h-1} \sum_{v=0}^{k_w-1} K(u,v) \cdot X_{pad}(i \cdot s + u,\; j \cdot s + v),
   $$
      scanning the input **without** flipping the kernel (the convention of deep learning frameworks, different from classical mathematical convolution).

6. **Activation:** Apply ReLU to each value: $a(i,j) = \max(0, z(i,j))$.

7. **Output dimensions:** $O_h = \lfloor (H + 2p - k_h)/s \rfloor + 1$ and $O_w = \lfloor (W + 2p - k_w)/s \rfloor + 1$.

8. **Output:** Print $O_h$ and $O_w$ on the first line, followed by $O_h$ lines with $O_w$ real values each (the output feature map, already with ReLU applied), formatted with 4 decimal places.

#### 📌 Computational Restrictions

* **One input channel, one output filter:** there is no need to handle multiple channels or multiple filters in this simplified version.
* **No kernel flipping:** implement cross-correlation, not the classical mathematical convolution with a flipped kernel — this is the operation that PyTorch (and most frameworks) calls "convolution".
* **Zero padding:** the $p$ pixels added to each border are always $0$.
* **Formatting:** all output values must have exactly 4 decimal places, even when the value is an integer (e.g., `2.0000`).

#### 🧠 Theoretical Foundation

| Element | Role in the convolutional layer |
|---|---|
| Kernel $K$ | Parameters learned by gradient, analogous to the coefficients of a fixed filter from Chapter 3, but adjusted by backpropagation |
| Bias $b$ | Learned offset, added after the correlation — allows the neuron to "fire" even with null input |
| Padding | Controls the spatial output dimension and prevents the loss of information at the borders at each layer |
| Stride | Controls the sliding step; values $> 1$ reduce spatial resolution, as a form of downsampling embedded within the convolution itself |
| ReLU | Introduces non-linearity after the linear combination, exactly as in the "Activation Function" Section |

#### 🧩 `morph.py` Methods That May Help

* `mm.readImg(h, w, dtype='float')` — directly reads an $h \times w$ matrix of real values from standard input, saving manual parsing of the feature map and the kernel.
* `mm.correlacao0(f, kernel, bias)` — implements the same cross-correlation sum + bias that you will compute by hand, but **without** support for padding or stride, and converts the result to `uint8` (truncating negative and decimal values). It may serve as a conceptual reference or to check the simplest case ($p=0$, $s=1$), but it does not replace your complete implementation — which must preserve sign, decimal places, padding, stride, and ReLU.

#### 📦 Input and Output Specification (VPL)

**Input:**

* Line 1: Integers $H$ and $W$.
* Next $H$ lines: $W$ real values each (input map).
* Next line: Integers $k_h$ and $k_w$.
* Next $k_h$ lines: $k_w$ real values each (kernel).
* Next line: Real $b$ (bias).
* Next line: Integers $p$ and $s$.

**Output:**

* Line 1: Integers $O_h$ and $O_w$.
* Next $O_h$ lines: $O_w$ real values each, with 4 decimal places.

#### 📌 Examples

| Input | Output | Observation |
|---|---|---|
| 3 3<br>1 2 0<br>0 1 2<br>1 0 1<br>2 2<br>1 1<br>1 1<br>-2<br>0 1 | 2 2<br>2.0000 3.0000<br>0.0000 2.0000 | Padding 0, stride 1: $2\times2$ output without padding. |
| 3 3<br>1 2 0<br>0 1 2<br>1 0 1<br>2 2<br>1 0<br>0 1<br>0<br>1 2 | 2 2<br>1.0000 0.0000<br>1.0000 2.0000 | Padding 1, stride 2: input padded with zeros before the correlation. |

In [ ]:
from IPython.display import HTML
HTML('''
<div id="sim-ep0901" style="background-color:#fef9ef;border-radius:18px;border:1px solid #ede6d8;overflow:hidden;margin-top:20px;font-family:sans-serif;">
  <div style="background:#f3efe6;padding:8px 16px;font-size:12px;color:#5e5a4a;border-bottom:1px solid #e9dfcf;display:flex;justify-content:space-between;align-items:center;">
    <span>🎮 Simulator: Manual 2D Convolution</span>
    <span style="background:#e8e0cf;border-radius:40px;padding:2px 10px;font-weight:600;font-size:10px;">🟢 cross-correlation + bias + ReLU</span>
  </div>
  <div style="padding:20px;background:white;">
    <p style="font-size:11px;color:#777;margin-bottom:14px;text-align:center;">Fixed 4×4 input, fixed 2×2 kernel (highlighted in blue) &mdash; adjust <em>padding</em> (p), <em>stride</em> (s) and bias (b), exactly the parameters EP09_01 asks for as input, and see how they change the output size and values.</p>

    <div style="display:flex;gap:24px;justify-content:center;flex-wrap:wrap;margin-bottom:6px;">
      <div>
        <div style="font-size:11px;font-weight:bold;color:#2980b9;margin-bottom:4px;text-align:center;">Padding (p)</div>
        <div id="ep0901_pad_btns" style="display:flex;gap:4px;"></div>
      </div>
      <div>
        <div style="font-size:11px;font-weight:bold;color:#2980b9;margin-bottom:4px;text-align:center;">Stride (s)</div>
        <div id="ep0901_stride_btns" style="display:flex;gap:4px;"></div>
      </div>
      <div>
        <div style="font-size:11px;font-weight:bold;color:#2980b9;margin-bottom:4px;text-align:center;">Bias (b)</div>
        <input id="ep0901_bias" type="number" step="0.5" value="0.5" style="width:70px;font-family:monospace;text-align:center;border:1px solid #ccc;border-radius:6px;padding:3px;">
      </div>
    </div>

    <div id="ep0901_formula" style="text-align:center;font-family:monospace;font-size:11px;color:#8a6d3b;background:#fcf5e6;border:1px solid #f0e2bf;border-radius:8px;padding:6px 10px;margin:10px auto 16px;max-width:520px;"></div>

    <div style="background:#fafafa;border:1px solid #ddd;border-radius:12px;padding:20px;margin-bottom:20px;">
      <div style="display:flex;justify-content:space-between;margin-bottom:8px;">
        <label style="font-size:12px;font-weight:bold;color:#2980b9;">Output position (i,j)</label>
        <span id="ep0901_vl" style="font-family:monospace;font-weight:bold;color:#2980b9;">(0,0)</span>
      </div>
      <input id="ep0901_sl" style="width:100%;accent-color:#2980b9;" max="8" min="0" step="1" type="range" value="0">
    </div>

    <div style="display:flex;gap:30px;justify-content:center;flex-wrap:wrap;align-items:flex-start;">
      <div>
        <div style="font-size:11px;color:#777;text-align:center;margin-bottom:6px;">Padded input X (with padding)</div>
        <div id="ep0901_grid" style="display:grid;gap:3px;"></div>
        <div style="display:flex;gap:10px;justify-content:center;margin-top:8px;font-size:9px;color:#888;">
          <span><span style="display:inline-block;width:9px;height:9px;background:#f3f4f6;border:1px solid #e5e7eb;border-radius:2px;vertical-align:middle;"></span> original</span>
          <span><span style="display:inline-block;width:9px;height:9px;background:#f5f5f5;border:1px dashed #ccc;border-radius:2px;vertical-align:middle;"></span> padding (0)</span>
          <span><span style="display:inline-block;width:9px;height:9px;background:#fff3cd;border:2px solid #f0ad4e;border-radius:2px;vertical-align:middle;"></span> current window</span>
        </div>
      </div>
      <div>
        <div style="font-size:11px;color:#777;text-align:center;margin-bottom:6px;">Kernel K (2×2)</div>
        <div id="ep0901_kernel" style="display:grid;grid-template-columns:repeat(2,44px);gap:3px;"></div>
      </div>
      <div>
        <div style="font-size:11px;color:#777;text-align:center;margin-bottom:6px;">Output Y = ReLU(X⊛K + b)</div>
        <div id="ep0901_out" style="display:grid;gap:3px;"></div>
      </div>
    </div>

    <div style="margin-top:14px;text-align:center;">
      <button id="ep0901_reset" style="font-family:sans-serif;font-size:11px;color:#5e5a4a;background:#f3efe6;border:1px solid #e0d7c4;border-radius:20px;padding:4px 14px;cursor:pointer;">↺ reset exploration</button>
    </div>
    <div id="ep0901_debug" style="margin-top:16px;background:#e3f2fd;border-radius:8px;padding:10px;border:1px solid #bbdefb;font-family:monospace;font-size:11px;color:#1565c0;text-align:center;"></div>
    <p style="font-size:10px;color:#999;margin-top:10px;text-align:center;">💡 Each slider position reveals one cell of the output matrix. Go through all positions to complete the output map. Changing p, s, or b resets the exploration because the output map changes in size and/or values.</p>
  </div>
</div>
<script>
(function(){
  function init(root){
    if(!root || root.dataset.init) return;
    root.dataset.init = "1";

    var H = 4, W = 4, kh = 2, kw = 2;
    var X = [[1,3,2,0],[0,1,4,1],[2,0,1,3],[1,2,0,1]];
    var K = [[1,0],[0,-1]];

    var state = { p: 0, s: 1, bias: 0.5 };
    var visited = {};

    var slEl = root.querySelector('#ep0901_sl');
    var vlEl = root.querySelector('#ep0901_vl');
    var gridEl = root.querySelector('#ep0901_grid');
    var kernelEl = root.querySelector('#ep0901_kernel');
    var outEl = root.querySelector('#ep0901_out');
    var dbg = root.querySelector('#ep0901_debug');
    var formulaEl = root.querySelector('#ep0901_formula');
    var resetBtn = root.querySelector('#ep0901_reset');
    var padBtnsEl = root.querySelector('#ep0901_pad_btns');
    var strideBtnsEl = root.querySelector('#ep0901_stride_btns');
    var biasInput = root.querySelector('#ep0901_bias');

    kernelEl.innerHTML = '';
    for(var u=0; u<kh; u++) for(var v=0; v<kw; v++){
      var kd = document.createElement('div');
      kd.style.cssText = 'width:44px;height:44px;display:flex;align-items:center;justify-content:center;background:#bbdefb;border:1px solid #64b5f6;border-radius:6px;font-family:monospace;font-weight:bold;color:#0d47a1;';
      kd.textContent = K[u][v];
      kernelEl.appendChild(kd);
    }

    function btnStyle(active){
      return 'font-family:monospace;font-size:12px;font-weight:bold;width:34px;height:30px;border-radius:8px;cursor:pointer;' +
        (active ? 'background:#2980b9;color:white;border:1px solid #2471a3;' : 'background:#f3f4f6;color:#555;border:1px solid #ddd;');
    }

    function buildButtons(container, values, current, onPick){
      container.innerHTML = '';
      values.forEach(function(val){
        var b = document.createElement('button');
        b.textContent = val;
        b.style.cssText = btnStyle(val === current);
        b.addEventListener('click', function(){ onPick(val); });
        container.appendChild(b);
      });
    }

    function buildPadded(p){
      var size = H + 2*p;
      var Xp = [];
      for(var r=0; r<size; r++){
        var row = [];
        for(var c=0; c<size; c++){
          var origR = r-p, origC = c-p;
          var isPad = !(origR>=0 && origR<H && origC>=0 && origC<W);
          row.push({ val: isPad ? 0 : X[origR][origC], pad: isPad });
        }
        Xp.push(row);
      }
      return Xp;
    }

    function computeAll(p, s, bias){
      var Xp = buildPadded(p);
      var size = H + 2*p;
      var Oh = Math.floor((size - kh)/s) + 1;
      var Ow = Math.floor((size - kw)/s) + 1;
      var vals = [];
      for(var i=0; i<Oh; i++){
        vals.push([]);
        for(var j=0; j<Ow; j++){
          var soma = 0;
          for(var u=0; u<kh; u++) for(var v=0; v<kw; v++) soma += K[u][v]*Xp[i*s+u][j*s+v].val;
          var z = soma + bias;
          var a = Math.max(0, z);
          vals[i].push({ soma: soma, z: z, a: a });
        }
      }
      return { Xp: Xp, size: size, Oh: Oh, Ow: Ow, vals: vals };
    }

    var model = null;

    function rebuildModel(resetPosition){
      model = computeAll(state.p, state.s, state.bias);
      gridEl.style.gridTemplateColumns = 'repeat(' + model.size + ', ' + Math.min(44, Math.floor(360/model.size)) + 'px)';
      outEl.style.gridTemplateColumns = 'repeat(' + model.Ow + ', 44px)';
      var maxPos = model.Oh*model.Ow - 1;
      slEl.max = maxPos;
      if(resetPosition){ slEl.value = 0; visited = {}; }
      else if(parseInt(slEl.value) > maxPos){ slEl.value = maxPos; }
      formulaEl.textContent = 'O_h = ⌊(' + H + ' + 2·' + state.p + ' − ' + kh + ')/' + state.s + '⌋ + 1 = ' + model.Oh +
        '    O_w = ⌊(' + W + ' + 2·' + state.p + ' − ' + kw + ')/' + state.s + '⌋ + 1 = ' + model.Ow;
      render();
    }

    function render(){
      var pos = parseInt(slEl.value);
      var i = Math.floor(pos/model.Ow), j = pos % model.Ow;
      var key = i+','+j;
      visited[key] = true;
      vlEl.textContent = '('+i+','+j+')';

      var winRowStart = i*state.s, winColStart = j*state.s;

      gridEl.innerHTML = '';
      var cellPx = Math.min(44, Math.floor(360/model.size));
      for(var r=0; r<model.size; r++){
        for(var c=0; c<model.size; c++){
          var cell = model.Xp[r][c];
          var dentroJanela = (r>=winRowStart && r<winRowStart+kh && c>=winColStart && c<winColStart+kw);
          var d = document.createElement('div');
          var base = 'width:'+cellPx+'px;height:'+cellPx+'px;display:flex;align-items:center;justify-content:center;border-radius:5px;font-family:monospace;font-size:11px;';
          if(dentroJanela){
            base += 'background:#fff3cd;border:2px solid #f0ad4e;font-weight:bold;color:#7a5c00;';
          } else if(cell.pad){
            base += 'background:#f5f5f5;border:1px dashed #ccc;color:#bbb;';
          } else {
            base += 'background:#f3f4f6;border:1px solid #e5e7eb;color:#555;';
          }
          d.style.cssText = base;
          d.textContent = cell.val;
          gridEl.appendChild(d);
        }
      }

      var cur = model.vals[i][j];

      outEl.innerHTML = '';
      for(var oi=0; oi<model.Oh; oi++){
        for(var oj=0; oj<model.Ow; oj++){
          var isCurrent = (oi===i && oj===j);
          var wasVisited = !!visited[oi+','+oj];
          var od = document.createElement('div');
          var style = 'width:44px;height:44px;display:flex;align-items:center;justify-content:center;border-radius:6px;font-family:monospace;font-size:11px;font-weight:bold;';
          if(isCurrent){
            style += 'background:#fff3cd;border:2px solid #f0ad4e;color:#7a5c00;';
          } else if(wasVisited){
            style += 'background:#e8f5e9;border:1px solid #a5d6a7;color:#2e7d32;';
          } else {
            style += 'background:#f3f4f6;border:1px dashed #ccc;color:#bbb;';
          }
          od.style.cssText = style;
          od.textContent = wasVisited ? model.vals[oi][oj].a.toFixed(2) : '?';
          outEl.appendChild(od);
        }
      }

      var totalVisitadas = Object.keys(visited).length;
      var totalPos = model.Oh*model.Ow;
      dbg.textContent = 'janela em ('+i+','+j+'), topo-esquerda em X_pad('+winRowStart+','+winColStart+')  |  soma(X⊙K)='+cur.soma.toFixed(2)+'  +  viés='+state.bias.toFixed(2)+'  =  z='+cur.z.toFixed(2)+'  →  ReLU(z)='+cur.a.toFixed(4)+'   ['+totalVisitadas+'/'+totalPos+' posições exploradas]';
    }

    function setPadding(val){
      state.p = val;
      buildButtons(padBtnsEl, [0,1,2], state.p, setPadding);
      buildButtons(strideBtnsEl, [1,2], state.s, setStride);
      rebuildModel(true);
    }
    function setStride(val){
      state.s = val;
      buildButtons(padBtnsEl, [0,1,2], state.p, setPadding);
      buildButtons(strideBtnsEl, [1,2], state.s, setStride);
      rebuildModel(true);
    }
    buildButtons(padBtnsEl, [0,1,2], state.p, setPadding);
    buildButtons(strideBtnsEl, [1,2], state.s, setStride);

    biasInput.addEventListener('change', function(){
      var v = parseFloat(biasInput.value);
      state.bias = isNaN(v) ? 0 : v;
      rebuildModel(true);
    });

    resetBtn.addEventListener('click', function(){
      visited = {};
      slEl.value = 0;
      render();
    });
    slEl.addEventListener('input', render);

    rebuildModel(true);
  }
  function tryInit(){
    var root = document.getElementById('sim-ep0901');
    if(root) init(root); else setTimeout(tryInit, 200);
  }
  tryInit();
})();
</script>
''')


**Figure 9.1:** EP09_01 Simulator: Manual 2D Convolution (cross-correlation + bias + ReLU, with adjustable *padding* and *stride*)


<figure id="fig-09-sim-ep0901">
  <img src="imagens/fig-09-sim-ep0901.png" alt=" EP09_01 Simulator: Manual 2D Convolution (cross-correlation + bias + ReLU, with adjustable *padding* and *stride*) " style="max-width:80%" />
  <figcaption><strong>Figure 9.1:</strong>  EP09_01 Simulator: Manual 2D Convolution (cross-correlation + bias + ReLU, with adjustable *padding* and *stride*) </figcaption>
</figure>

In [ ]:
%%writefile EP09_01.py
# Python code

In [ ]:
TestSuite("EP09_01.py").run()

### EP09_02 🟢 Manual *Pooling* (Maximum and Average)

Between convolutional blocks, the typical architecture of a CNN interleaves ***pooling*** layers, which reduce the spatial resolution of the feature map without introducing new trainable parameters — unlike convolution, *pooling* has no weights: it simply summarizes each window of the input into a single value, either by a maximum or by an average, exactly as formalized in the Section "*Pooling*".

You have been tasked with implementing this operation using a square sliding window, without partial overlap at the borders (only complete windows), supporting the two most common types: `max` (preserves the most salient value, typically used to retain strong edges and textures) and `avg` (smooths the region, preserving average intensity information).

#### 📋 Implementation Guidelines

1. **Input:** Read the dimensions $H \times W$ of the input feature map and its $H \times W$ real values.
2. **Window:** Read the integers $k$ (size of the square window $k \times k$) and $s$ (*stride*).
3. **Type:** Read a *string*, `max` or `avg`, indicating the type of *pooling*.
4. **No padding:** This operation **does not** use *padding*; windows that would exceed the input border are discarded.
5. **Calculation:** For each output position $(i,j)$, compute the maximum or the average of the $k \times k$ values of the corresponding window, starting at $(i \cdot s,\, j \cdot s)$.
6. **Output dimensions:** $O_h = \lfloor (H - k)/s \rfloor + 1$ and $O_w = \lfloor (W - k)/s \rfloor + 1$.
7. **Output:** Print $O_h$ and $O_w$ on the first line, followed by $O_h$ lines with $O_w$ real values each, formatted with 4 decimal places.

#### 📌 Computational Constraints

* **Square window:** $k \times k$, with no support for rectangular windows in this version.
* **No *padding*:** only windows fully contained within the input are considered — dimensions that "remain" are simply discarded.
* **`avg` uses real division:** the average is always $\text{sum}/k^2$, even when the result has many decimal places — round only in the final formatting, according to the general guideline of the chapter.
* **Formatting:** all output values with exactly 4 decimal places.

#### 🧠 Theoretical Foundation

| Element | Role in the architecture |
|---|---|
| Maximum *pooling* | Preserves the strongest activation of the window; common after convolutional layers to retain salient edges and textures |
| Average *pooling* | Smooths the region, preserving average intensity; common in final layers (*global average pooling*) |
| Absence of parameters | Differentiates *pooling* from convolution: reduces spatial resolution without additional training cost |
| Resolution reduction | Contributes to invariance to small translations and to the reduction of the computational cost of subsequent layers |

#### 🧩 Methods from `morph.py` that may help

`morph.py` does not implement *pooling* with subsampling directly, but two families of operations show the same idea from another perspective, useful for checking your intuition:

* `mm.dil(f, Bc)` / `mm.dil0(f, B)` — morphological dilation: replaces each pixel by the **maximum** of its neighborhood defined by the structuring element $B$ (e.g., `mm.sebox(n)` for a $(2n+1)\times(2n+1)$ window). It is, conceptually, a "*max-pooling* without subsampling" (produces an image of the same size, rather than a reduced one).
* `mm.blur(f, N)` — averaging smoothing in an $N \times N$ window, analogous to *avg-pooling*, also without resolution reduction.
* `mm.readImg(h, w, dtype='float')` — useful for reading the input map in floating point.

#### 📦 Input and Output Specification (VPL)

**Input:**

* Line 1: Integers $H$ and $W$.
* Next $H$ lines: $W$ real values each.
* Next line: Integers $k$ and $s$.
* Next line: `max` or `avg`.

**Output:**

* Line 1: Integers $O_h$ and $O_w$.
* Next $O_h$ lines: $O_w$ real values each, with 4 decimal places.

#### 📌 Examples

| Input | Output | Observation |
|---|---|---|
| 4 4<br>1 3 2 4<br>5 6 1 2<br>2 1 0 3<br>4 2 5 1<br>2 2<br>max | 2 2<br>6.0000 4.0000<br>4.0000 5.0000 | Maximum *pooling*, $2\times2$ window, *stride* 2. |
| 4 4<br>1 3 2 4<br>5 6 1 2<br>2 1 0 3<br>4 2 5 1<br>2 2<br>avg | 2 2<br>3.7500 2.2500<br>2.2500 2.2500 | Average *pooling* over the same windows. |

In [ ]:
from IPython.display import HTML
HTML('''
<div id="sim-ep0902" style="background-color:#fef9ef;border-radius:18px;border:1px solid #ede6d8;overflow:hidden;margin-top:20px;font-family:sans-serif;">
  <div style="background:#f3efe6;padding:8px 16px;font-size:12px;color:#5e5a4a;border-bottom:1px solid #e9dfcf;display:flex;justify-content:space-between;align-items:center;">
    <span>🎮 Simulator: Manual Pooling</span>
    <span style="background:#e8e0cf;border-radius:40px;padding:2px 10px;font-weight:600;font-size:10px;">🟢 no padding, full windows</span>
  </div>
  <div style="padding:20px;background:white;">
    <p style="font-size:11px;color:#777;margin-bottom:14px;text-align:center;">Fixed 4×4 input &mdash; adjust the window size (k), stride (s) and type, exactly the parameters that EP09_02 reads as input, and see how they change the size and values of the output.</p>

    <div style="display:flex;gap:24px;justify-content:center;flex-wrap:wrap;margin-bottom:6px;">
      <div>
        <div style="font-size:11px;font-weight:bold;color:#2980b9;margin-bottom:4px;text-align:center;">Window (k)</div>
        <div id="ep0902_k_btns" style="display:flex;gap:4px;"></div>
      </div>
      <div>
        <div style="font-size:11px;font-weight:bold;color:#2980b9;margin-bottom:4px;text-align:center;">Stride (s)</div>
        <div id="ep0902_s_btns" style="display:flex;gap:4px;"></div>
      </div>
      <div>
        <div style="font-size:11px;font-weight:bold;color:#2980b9;margin-bottom:4px;text-align:center;">Type</div>
        <div style="display:flex;gap:8px;">
          <button id="ep0902_max" style="cursor:pointer;padding:6px 14px;border-radius:20px;border:1px solid #f0ad4e;background:#fff3cd;color:#7a5c00;font-weight:bold;font-size:11px;">max</button>
          <button id="ep0902_avg" style="cursor:pointer;padding:6px 14px;border-radius:20px;border:1px solid #ddd;background:#f3f4f6;color:#555;font-weight:bold;font-size:11px;">avg</button>
        </div>
      </div>
    </div>

    <div id="ep0902_formula" style="text-align:center;font-family:monospace;font-size:11px;color:#8a6d3b;background:#fcf5e6;border:1px solid #f0e2bf;border-radius:8px;padding:6px 10px;margin:10px auto 16px;max-width:520px;"></div>

    <div style="background:#fafafa;border:1px solid #ddd;border-radius:12px;padding:20px;margin-bottom:20px;">
      <div style="display:flex;justify-content:space-between;margin-bottom:8px;">
        <label style="font-size:12px;font-weight:bold;color:#2980b9;">Output position (i,j)</label>
        <span id="ep0902_vl" style="font-family:monospace;font-weight:bold;color:#2980b9;">(0,0)</span>
      </div>
      <input id="ep0902_sl" style="width:100%;accent-color:#2980b9;" max="3" min="0" step="1" type="range" value="0">
    </div>

    <div style="display:flex;gap:30px;justify-content:center;flex-wrap:wrap;align-items:flex-start;">
      <div>
        <div style="font-size:11px;color:#777;text-align:center;margin-bottom:6px;">Input X (4×4)</div>
        <div id="ep0902_grid" style="display:grid;grid-template-columns:repeat(4,44px);gap:3px;"></div>
        <div style="display:flex;gap:10px;justify-content:center;margin-top:8px;font-size:9px;color:#888;">
          <span><span style="display:inline-block;width:9px;height:9px;background:#f3f4f6;border:1px solid #e5e7eb;border-radius:2px;vertical-align:middle;"></span> outside the window</span>
          <span><span style="display:inline-block;width:9px;height:9px;background:#fff3cd;border:2px solid #f0ad4e;border-radius:2px;vertical-align:middle;"></span> current window</span>
          <span><span style="display:inline-block;width:9px;height:9px;background:#eee;border:1px dashed #bbb;border-radius:2px;vertical-align:middle;"></span> discarded (leftover)</span>
        </div>
      </div>
      <div>
        <div style="font-size:11px;color:#777;text-align:center;margin-bottom:6px;">Output Y (pooling)</div>
        <div id="ep0902_out" style="display:grid;gap:3px;"></div>
      </div>
    </div>

    <div style="margin-top:14px;text-align:center;">
      <button id="ep0902_reset" style="font-family:sans-serif;font-size:11px;color:#5e5a4a;background:#f3efe6;border:1px solid #e0d7c4;border-radius:20px;padding:4px 14px;cursor:pointer;">↺ restart exploration</button>
    </div>
    <div id="ep0902_debug" style="margin-top:16px;background:#e3f2fd;border-radius:8px;padding:10px;border:1px solid #bbdefb;font-family:monospace;font-size:11px;color:#1565c0;text-align:center;"></div>
    <p style="font-size:10px;color:#999;margin-top:10px;text-align:center;">💡 Each slider position reveals a cell of the output matrix. Dashed-gray cells in the input are "leftovers" that no window reaches &mdash; note how this happens when (H&minus;k) is not a multiple of s. Changing k, s, or the type restarts the exploration.</p>
  </div>
</div>
<script>
(function(){
  function init(root){
    if(!root || root.dataset.init) return;
    root.dataset.init = "1";

    var H = 4, W = 4;
    var X = [[1,3,2,4],[5,6,1,2],[2,1,0,3],[4,2,5,1]];

    var state = { k: 2, s: 2, tipo: 'max' };
    var visited = {};

    var slEl = root.querySelector('#ep0902_sl');
    var vlEl = root.querySelector('#ep0902_vl');
    var gridEl = root.querySelector('#ep0902_grid');
    var outEl = root.querySelector('#ep0902_out');
    var dbg = root.querySelector('#ep0902_debug');
    var formulaEl = root.querySelector('#ep0902_formula');
    var resetBtn = root.querySelector('#ep0902_reset');
    var kBtnsEl = root.querySelector('#ep0902_k_btns');
    var sBtnsEl = root.querySelector('#ep0902_s_btns');
    var btnMax = root.querySelector('#ep0902_max');
    var btnAvg = root.querySelector('#ep0902_avg');

    function btnStyle(active){
      return 'font-family:monospace;font-size:12px;font-weight:bold;width:34px;height:30px;border-radius:8px;cursor:pointer;' +
        (active ? 'background:#2980b9;color:white;border:1px solid #2471a3;' : 'background:#f3f4f6;color:#555;border:1px solid #ddd;');
    }

    function buildButtons(container, values, current, onPick){
      container.innerHTML = '';
      values.forEach(function(val){
        var b = document.createElement('button');
        b.textContent = val;
        b.style.cssText = btnStyle(val === current);
        b.addEventListener('click', function(){ onPick(val); });
        container.appendChild(b);
      });
    }

    function estiloTipoBotoes(){
      btnMax.style.background = state.tipo==='max' ? '#fff3cd' : '#f3f4f6';
      btnMax.style.borderColor = state.tipo==='max' ? '#f0ad4e' : '#ddd';
      btnMax.style.color = state.tipo==='max' ? '#7a5c00' : '#555';
      btnAvg.style.background = state.tipo==='avg' ? '#fff3cd' : '#f3f4f6';
      btnAvg.style.borderColor = state.tipo==='avg' ? '#f0ad4e' : '#ddd';
      btnAvg.style.color = state.tipo==='avg' ? '#7a5c00' : '#555';
    }

    function computeAll(k, s, tipo){
      var Oh = Math.floor((H - k)/s) + 1;
      var Ow = Math.floor((W - k)/s) + 1;
      var vals = [];
      for(var i=0; i<Oh; i++){
        vals.push([]);
        for(var j=0; j<Ow; j++){
          var janela = [];
          for(var r=i*s; r<i*s+k; r++) for(var c=j*s; c<j*s+k; c++) janela.push(X[r][c]);
          var resultado = tipo === 'max'
            ? Math.max.apply(null, janela)
            : janela.reduce(function(a,b){return a+b;},0)/janela.length;
          vals[i].push({ janela: janela, resultado: resultado });
        }
      }
      return { Oh: Oh, Ow: Ow, vals: vals };
    }

    var model = null;

    function rebuildModel(resetPosition){
      model = computeAll(state.k, state.s, state.tipo);
      outEl.style.gridTemplateColumns = 'repeat(' + model.Ow + ', 48px)';
      var maxPos = model.Oh*model.Ow - 1;
      slEl.max = maxPos;
      if(resetPosition){ slEl.value = 0; visited = {}; }
      else if(parseInt(slEl.value) > maxPos){ slEl.value = maxPos; }
      formulaEl.textContent = 'O_h = ⌊(' + H + ' − ' + state.k + ')/' + state.s + '⌋ + 1 = ' + model.Oh +
        '    O_w = ⌊(' + W + ' − ' + state.k + ')/' + state.s + '⌋ + 1 = ' + model.Ow;
      estiloTipoBotoes();
      render();
    }

    function render(){
      var pos = parseInt(slEl.value);
      var i = Math.floor(pos/model.Ow), j = pos % model.Ow;
      var key = i+','+j;
      visited[key] = true;
      vlEl.textContent = '('+i+','+j+')';

      var winRowStart = i*state.s, winColStart = j*state.s;
      var alcancavel = []; // marca quais células de X são alcançadas por ALGUMA janela válida
      for(var r=0;r<H;r++){ alcancavel.push(new Array(W).fill(false)); }
      for(var oi=0; oi<model.Oh; oi++){
        for(var oj=0; oj<model.Ow; oj++){
          for(var r=oi*state.s; r<oi*state.s+state.k; r++)
            for(var c=oj*state.s; c<oj*state.s+state.k; c++)
              alcancavel[r][c] = true;
        }
      }

      gridEl.innerHTML = '';
      for(var r=0; r<H; r++){
        for(var c=0; c<W; c++){
          var dentroJanela = (r>=winRowStart && r<winRowStart+state.k && c>=winColStart && c<winColStart+state.k);
          var d = document.createElement('div');
          var base = 'width:44px;height:44px;display:flex;align-items:center;justify-content:center;border-radius:6px;font-family:monospace;font-size:13px;';
          if(dentroJanela){
            base += 'background:#fff3cd;border:2px solid #f0ad4e;font-weight:bold;color:#7a5c00;';
          } else if(!alcancavel[r][c]){
            base += 'background:#eee;border:1px dashed #bbb;color:#999;';
          } else {
            base += 'background:#f3f4f6;border:1px solid #e5e7eb;color:#555;';
          }
          d.style.cssText = base;
          d.textContent = X[r][c];
          gridEl.appendChild(d);
        }
      }

      var cur = model.vals[i][j];

      outEl.innerHTML = '';
      for(var oi2=0; oi2<model.Oh; oi2++){
        for(var oj2=0; oj2<model.Ow; oj2++){
          var isCurrent = (oi2===i && oj2===j);
          var wasVisited = !!visited[oi2+','+oj2];
          var od = document.createElement('div');
          var style = 'width:48px;height:48px;display:flex;align-items:center;justify-content:center;border-radius:6px;font-family:monospace;font-size:11px;font-weight:bold;';
          if(isCurrent){
            style += 'background:#fff3cd;border:2px solid #f0ad4e;color:#7a5c00;';
          } else if(wasVisited){
            style += 'background:#e8f5e9;border:1px solid #a5d6a7;color:#2e7d32;';
          } else {
            style += 'background:#f3f4f6;border:1px dashed #ccc;color:#bbb;';
          }
          od.style.cssText = style;
          od.textContent = wasVisited ? model.vals[oi2][oj2].resultado.toFixed(2) : '?';
          outEl.appendChild(od);
        }
      }

      var totalVisitadas = Object.keys(visited).length;
      var totalPos = model.Oh*model.Ow;
      dbg.textContent = 'janela=['+cur.janela.join(', ')+']  |  tipo='+state.tipo+'  →  resultado='+cur.resultado.toFixed(4)+'   ['+totalVisitadas+'/'+totalPos+' posições exploradas]';
    }

    function setK(val){
      state.k = val;
      buildButtons(kBtnsEl, [1,2,3,4], state.k, setK);
      buildButtons(sBtnsEl, [1,2,3], state.s, setS);
      rebuildModel(true);
    }
    function setS(val){
      state.s = val;
      buildButtons(kBtnsEl, [1,2,3,4], state.k, setK);
      buildButtons(sBtnsEl, [1,2,3], state.s, setS);
      rebuildModel(true);
    }
    buildButtons(kBtnsEl, [1,2,3,4], state.k, setK);
    buildButtons(sBtnsEl, [1,2,3], state.s, setS);

    btnMax.addEventListener('click', function(){ state.tipo='max'; rebuildModel(true); });
    btnAvg.addEventListener('click', function(){ state.tipo='avg'; rebuildModel(true); });

    resetBtn.addEventListener('click', function(){
      visited = {};
      slEl.value = 0;
      render();
    });
    slEl.addEventListener('input', render);

    rebuildModel(true);
  }
  function tryInit(){
    var root = document.getElementById('sim-ep0902');
    if(root) init(root); else setTimeout(tryInit, 200);
  }
  tryInit();
})();
</script>
''')


**Figure 9.2:** Simulator EP09_02: Manual Pooling (max vs. mean, with adjustable window k and stride s)


<figure id="fig-09-sim-ep0902">
  <img src="imagens/fig-09-sim-ep0902.png" alt=" Simulator EP09_02: Manual Pooling (max vs. mean, with adjustable window k and stride s) " style="max-width:80%" />
  <figcaption><strong>Figure 9.2:</strong>  Simulator EP09_02: Manual Pooling (max vs. mean, with adjustable window k and stride s) </figcaption>
</figure>

In [ ]:
%%writefile EP09_02.py
# Python code

In [ ]:
TestSuite("EP09_02.py").run()

### EP09_03 🟡 Counting Trainable Parameters of a CNN

This EP formalizes the counting of trainable parameters of a *CNN*. Given the textual description of a small architecture, composed of convolutional, pooling, and fully connected layers, determine, for each layer, the number of trainable parameters and the network total.

The architecture must be interpreted **sequentially**: the output of a convolutional layer becomes the input of the next compatible layer. Thus, the number of channels produced by a `CONV` layer determines the number of input channels (`cin`) of the following convolutional layer.

In a convolutional layer, it is important to distinguish **input channels** and **output channels**:

* $c_{in}$ (*channels in*) is the number of **channels entering the layer**. A grayscale image has $c_{in}=1$, while an RGB image has $c_{in}=3$. In an intermediate convolutional layer, `cin` is typically equal to the number of channels produced by the preceding `CONV` layer.
* $c_{out}$ (*channels out*) is the number of **channels produced by the layer**. It equals the number of filters used. Therefore, if a layer has 16 filters, it produces $c_{out}=16$ channels.

For example, consider the sequence:

```text
CONV 3 3 1 8 1
POOL
CONV 3 3 8 16 1
POOL
FC 784 10 1
```

The first convolution receives an image with one channel and produces 8 channels. After pooling, the second convolution receives those 8 channels and produces 16 channels. The `POOL` layer does not alter the number of channels; it may only reduce the spatial dimensions. The `FC` layer receives the number of inputs specified in its own description.

Each convolutional filter has dimensions

$$
k_h \times k_w \times c_{in}.
$$

Thus, a layer with $c_{out}$ filters has

$$
k_h \cdot k_w \cdot c_{in} \cdot c_{out}
$$

weights. If there is a bias, one parameter is added for each filter, totaling an additional $c_{out}$ parameters.

The central point of this exercise is to observe that the number of parameters in a convolutional layer **does not depend on the spatial dimensions** ($H \times W$) of the feature map. This occurs due to **weight sharing**: the same filter is reused at different positions of the input.

#### 📋 Implementation Guidelines

1. **Input:** Read the integer $L$ (number of layers in the architecture, in the order they are applied).

2. **Layers:** Read $L$ lines, each describing a layer in one of three formats:

   * `CONV kh kw cin cout bias` — convolutional layer with *kernel* $k_h \times k_w$, $c_{in}$ input channels, $c_{out}$ output channels, and `bias` (0 or 1), indicating whether there is a bias per filter;
   * `POOL` — pooling layer (max or average), which has no trainable parameters and preserves the number of channels;
   * `FC in out bias` — fully connected layer with `in` inputs, `out` outputs, and `bias` (0 or 1), indicating whether there is a bias per neuron.

3. **Consistency between `CONV` layers:** in a sequence of convolutional layers, the `cin` of a layer must correspond to the `cout` of the preceding convolutional layer. A `POOL` layer does not alter this number of channels.

   For example:

   ```text
   CONV 3 3 1 8 1
   POOL
   CONV 3 3 8 16 1
   ```

   The first `CONV` produces 8 channels, which are received by the second `CONV`. Therefore, in the second layer, `cin=8` and `cout=16`.

4. **Parameters of a `CONV` layer:**

   Each of the $c_{out}$ filters has $k_h \cdot k_w \cdot c_{in}$ weights. Therefore,

   $$
   P_{\mathrm{CONV}} =
   k_h \cdot k_w \cdot c_{in} \cdot c_{out}
   +
   c_{out}\cdot\text{bias}.
   $$

5. **Parameters of an `FC` layer:**

   $$
   P_{\mathrm{FC}} = 
   \text{in}\cdot\text{out}
   +
   \text{out}\cdot\text{bias}.
   $$

6. **Parameters of a `POOL` layer:** always $0$.

7. **Network total:** sum the trainable parameters of all layers.

8. **Output:** For each layer, in reading order, print `Camada i: P`, where $i$ starts at $1$ and $P$ is the number of parameters for that layer. At the end, print `Total: T`.

#### 📐 Example to understand `cin` and `cout`

Consider the sequence:

```text
CONV 3 3 1 8 1
POOL
CONV 3 3 8 16 1
```

In the first layer:

* `cin=1`: one channel enters;
* `cout=8`: there are 8 filters and, therefore, 8 channels exit.

Each filter has

$$
3\cdot3\cdot1=9
$$

weights. Since there are 8 filters:

$$
9\cdot8=72
$$

weights. With one bias per filter:

$$
72+8=80.
$$

In the second layer:

* `cin=8`: the 8 channels produced by the first `CONV` enter;
* `cout=16`: there are 16 filters and, therefore, 16 channels exit.

Each filter has

$$
3\cdot3\cdot8=72
$$

weights. Since there are 16 filters:

$$
72\cdot16=1152
$$

weights. With 16 biases:

$$
1152+16=1168.
$$

Thus, the two layers have, respectively, **80** and **1168 trainable parameters**.

Note that `cout` **is not** $cin$ multiplied by the number of filters. The number of filters is exactly `cout`: each filter combines all input channels and produces **a single output channel**.

#### 📌 Computational Constraints

* **Independence from spatial dimension:** the input does not provide $H \times W$. The count for a `CONV` layer depends only on `kh`, `kw`, `cin`, and `cout`.
* **Channel consistency:** for two consecutive `CONV` layers, the `cin` of the second must equal the `cout` of the first. A `POOL` layer preserves the number of channels.
* **`bias` always 0 or 1:** multiply the bias term directly by this value.
* **`POOL` layers with no additional arguments:** the line contains only the word `POOL`.
* **`FC` layers:** the number of inputs `in` is provided explicitly. It is not necessary to compute the spatial dimensions produced by previous layers.
* All numerical input values are non-negative integers.

#### 🧠 Theoretical Foundation

| Element                  | Role in parameter counting                                                                                     |
| ------------------------ | -------------------------------------------------------------------------------------------------------------- |
| $c_{in}$                 | Number of channels received by the layer                                                                        |
| $c_{out}$                | Number of filters and, therefore, of channels produced by the layer                                             |
| Convolutional filter      | Each filter has $k_h \cdot k_w \cdot c_{in}$ weights and produces one output channel                            |
| Weight sharing           | The same filter is reused at different input positions, making the count independent of $H \times W$            |
| Bias                     | A single additional parameter per filter (`CONV`) or per neuron (`FC`)                                          |
| *Pooling*                | May alter $H \times W$, but has no trainable parameters and preserves the number of channels                    |
| `FC` layer               | Has one weight for each combination of input and output neuron                                                  |

#### 🧩 `morph.py` methods that may help

This exercise is purely arithmetic and does not directly use functions from `morph.py`. The count can, however, be checked in a real architecture implemented in PyTorch using:

```python
sum(p.numel() for p in modelo.parameters())
```

This expression counts the model parameters, including weights and biases.

#### 📦 Input and Output Specification (VPL)

**Input:**

* Line 1: Integer $L$.
* Next $L$ lines: description of each layer, in the format `CONV kh kw cin cout bias`, `POOL`, or `FC in out bias`.

**Output:**

* $L$ lines in the format `Camada i: P`.
* Last line: `Total: T`.

#### 📌 Examples

| Input                                                                                | Output                                                                                                         | Observation                                                                                                        |
| ------------------------------------------------------------------------------------ | -------------------------------------------------------------------------------------------------------------- | ------------------------------------------------------------------------------------------------------------------ |
| 3<br>CONV 3 3 1 8 1<br>POOL<br>FC 1352 10 1                                         | Camada 1: 80<br>Camada 2: 0<br>Camada 3: 13530<br>Total: 13610                                                 | Simple network with one convolution, *pooling*, and classification layer.                                          |
| 5<br>CONV 3 3 1 8 1<br>POOL<br>CONV 3 3 8 16 1<br>POOL<br>FC 400 10 1                | Camada 1: 80<br>Camada 2: 0<br>Camada 3: 1168<br>Camada 4: 0<br>Camada 5: 4010<br>Total: 5258                  | Small CNN with two convolutions, two *poolings*, and one fully connected layer.                                    |
| 6<br>CONV 3 3 1 8 1<br>POOL<br>CONV 3 3 8 16 1<br>POOL<br>FC 256 32 1<br>FC 32 10 1 | Camada 1: 80<br>Camada 2: 0<br>Camada 3: 1168<br>Camada 4: 0<br>Camada 5: 8224<br>Camada 6: 330<br>Total: 9802 | Small CNN with two convolutions, intermediate *pooling*, and two fully connected layers for classification.        |

In [ ]:
from IPython.display import HTML
HTML('''
<div id="sim-ep0903" style="background-color:#fef9ef;border-radius:18px;border:1px solid #ede6d8;overflow:hidden;margin-top:20px;font-family:sans-serif;">
  <div style="background:#f3efe6;padding:8px 16px;font-size:12px;color:#5e5a4a;border-bottom:1px solid #e9dfcf;display:flex;justify-content:space-between;align-items:center;">
    <span>🎮 Simulator: Parameter Counting</span>
    <span style="background:#e8e0cf;border-radius:40px;padding:2px 10px;font-weight:600;font-size:10px;">🟡 weight sharing</span>
  </div>
  <div style="padding:16px;background:white;">
    
    <!-- Legenda compacta -->
    <div style="background:#f8f9fa;border:1px solid #dee2e6;border-radius:8px;padding:6px 10px;margin-bottom:10px;display:flex;gap:12px;justify-content:center;font-size:10px;flex-wrap:wrap;">
      <span style="display:flex;align-items:center;gap:3px;">
        <span style="width:12px;height:12px;background:#4a90e2;border:2px solid #2c5f8a;border-radius:2px;display:inline-block;"></span>
        <b>CONV</b> Blue block
      </span>
      <span style="display:flex;align-items:center;gap:3px;">
        <span style="width:12px;height:12px;background:#50e3c2;border:2px solid #2c8a6e;border-radius:50%;display:inline-block;"></span>
        <b>POOL</b> Green cylinder
      </span>
      <span style="display:flex;align-items:center;gap:3px;">
        <span style="width:12px;height:12px;background:#f5a623;border:2px solid #b87d1a;display:inline-block;transform:rotate(45deg);"></span>
        <b>FC</b> Orange diamond
      </span>
      <span style="display:flex;align-items:center;gap:3px;">
        <span style="width:12px;height:12px;background:#ff6b6b;border:2px solid #cc4444;display:inline-block;"></span>
        <b>BATCH</b> Red stack
      </span>
      <span style="display:flex;align-items:center;gap:3px;color:#666;">
        🖱️ Drag to move layers
      </span>
    </div>
    
    <!-- Controles compactos -->
    <div style="background:#fafafa;border:1px solid #ddd;border-radius:10px;padding:12px;margin-bottom:12px;">
      <div style="display:grid;grid-template-columns:repeat(4,1fr);gap:10px;margin-bottom:8px;">
        <div>
          <div style="display:flex;justify-content:space-between;font-size:10px;margin-bottom:2px;">
            <label style="font-weight:bold;color:#2980b9;">H×W</label>
            <span id="ep0903_hw_v" style="font-family:monospace;font-weight:bold;color:#2980b9;font-size:9px;">32×32</span>
          </div>
          <input id="ep0903_hw" style="width:100%;accent-color:#2980b9;height:4px;" max="64" min="8" step="2" type="range" value="32">
        </div>
        <div>
          <div style="display:flex;justify-content:space-between;font-size:10px;margin-bottom:2px;">
            <label style="font-weight:bold;color:#2980b9;">Channels</label>
            <span id="ep0903_cin_v" style="font-family:monospace;font-weight:bold;color:#2980b9;font-size:9px;">1</span>
          </div>
          <input id="ep0903_cin" style="width:100%;accent-color:#2980b9;height:4px;" max="3" min="1" step="1" type="range" value="1">
        </div>
        <div>
          <div style="display:flex;justify-content:space-between;font-size:10px;margin-bottom:2px;">
            <label style="font-weight:bold;color:#2980b9;">Batch</label>
            <span id="ep0903_batch_v" style="font-family:monospace;font-weight:bold;color:#2980b9;font-size:9px;">4</span>
          </div>
          <input id="ep0903_batch" style="width:100%;accent-color:#2980b9;height:4px;" max="16" min="1" step="1" type="range" value="4">
        </div>
        <div>
          <div style="display:flex;justify-content:space-between;font-size:10px;margin-bottom:2px;">
            <label style="font-weight:bold;color:#2980b9;">Layers</label>
            <span id="ep0903_nlayers_v" style="font-family:monospace;font-weight:bold;color:#2980b9;font-size:9px;">3</span>
          </div>
          <input id="ep0903_nlayers" style="width:100%;accent-color:#2980b9;height:4px;" max="6" min="1" step="1" type="range" value="3">
        </div>
      </div>
      
      <!-- Configuração das camadas compacta -->
      <div id="ep0903_layers_config" style="margin-bottom:8px;display:flex;flex-wrap:wrap;gap:6px;">
        <!-- Gerado dinamicamente -->
      </div>
      
      <div style="display:flex;gap:12px;align-items:center;font-size:10px;">
        <label style="display:flex;align-items:center;gap:4px;cursor:pointer;">
          <input id="ep0903_bias" type="checkbox" checked style="accent-color:#2980b9;width:14px;height:14px;">
          <span style="font-weight:bold;color:#2980b9;">Use bias</span>
        </label>
      </div>
    </div>
    
    <!-- Visualização 3D -->
    <div style="position:relative;background:linear-gradient(135deg, #1e3c72 0%, #2a5298 100%);border-radius:10px;padding:16px;margin-bottom:12px;min-height:400px;">
      <div style="position:absolute;top:8px;left:10px;color:white;font-weight:bold;font-size:14px;text-shadow:2px 2px 4px rgba(0,0,0,0.5);z-index:10;">
        🧠 3D Visualization
      </div>
      
      <div style="position:absolute;top:8px;right:8px;display:flex;gap:4px;z-index:10;">
        <button id="ep0903_pause_btn" style="background:rgba(255,255,255,0.2);border:1px solid rgba(255,255,255,0.4);color:white;padding:4px 8px;border-radius:16px;cursor:pointer;font-size:10px;font-weight:bold;backdrop-filter:blur(5px);transition:all 0.3s;">
          ⏸️ Pause
        </button>
        <button id="ep0903_reset_btn" style="background:rgba(255,255,255,0.2);border:1px solid rgba(255,255,255,0.4);color:white;padding:4px 8px;border-radius:16px;cursor:pointer;font-size:10px;font-weight:bold;backdrop-filter:blur(5px);transition:all 0.3s;">
          🔄 Reset
        </button>
        <button id="ep0903_auto_layout_btn" style="background:rgba(255,255,255,0.2);border:1px solid rgba(255,255,255,0.4);color:white;padding:4px 8px;border-radius:16px;cursor:pointer;font-size:10px;font-weight:bold;backdrop-filter:blur(5px);transition:all 0.3s;">
          📐 Auto
        </button>
      </div>
      
      <canvas id="ep0903_canvas" style="width:100%;height:340px;display:block;cursor:grab;"></canvas>
      
      <div style="position:absolute;bottom:6px;right:8px;color:white;font-size:9px;background:rgba(0,0,0,0.5);padding:3px 8px;border-radius:14px;">
        🖱️ Drag layers | Scroll zoom | P to pause
      </div>
    </div>
    
    <!-- Resumo compacto -->
    <div id="ep0903_summary" style="background:#e3f2fd;border-radius:6px;padding:8px;border:1px solid #bbdefb;font-family:monospace;font-size:10px;color:#1565c0;line-height:1.5;">
    </div>
  </div>
</div>

<script>
(function(){
  function init(root){
    if(!root || root.dataset.init) return;
    root.dataset.init = "1";
    
    // Elementos DOM
    var hwEl = root.querySelector('#ep0903_hw'), hwvEl = root.querySelector('#ep0903_hw_v');
    var cinEl = root.querySelector('#ep0903_cin'), cinvEl = root.querySelector('#ep0903_cin_v');
    var batchEl = root.querySelector('#ep0903_batch'), batchvEl = root.querySelector('#ep0903_batch_v');
    var nlayersEl = root.querySelector('#ep0903_nlayers'), nlayersvEl = root.querySelector('#ep0903_nlayers_v');
    var layersConfigEl = root.querySelector('#ep0903_layers_config');
    var biasEl = root.querySelector('#ep0903_bias');
    var summaryEl = root.querySelector('#ep0903_summary');
    var canvas = root.querySelector('#ep0903_canvas');
    var ctx = canvas.getContext('2d');
    var pauseBtn = root.querySelector('#ep0903_pause_btn');
    var resetBtn = root.querySelector('#ep0903_reset_btn');
    var autoLayoutBtn = root.querySelector('#ep0903_auto_layout_btn');
    
    // Estado da visualização
    var rotationX = -0.3;
    var rotationY = 0.5;
    var zoom = 1;
    var isDragging = false;
    var isDraggingLayer = false;
    var selectedLayer = null;
    var lastX = 0;
    var lastY = 0;
    var autoRotate = true;
    var isPaused = false;
    var lastInteractionTime = Date.now();
    var animationId = null;
    var time = 0;
    
    // Posições das camadas
    var layerPositions = [];
    var batchPosition = { x: -6, y: -0.5, z: 0 };
    
    function resizeCanvas() {
      canvas.width = canvas.clientWidth;
      canvas.height = canvas.clientHeight;
    }
    resizeCanvas();
    window.addEventListener('resize', resizeCanvas);
    
    function autoLayout() {
      var nlayers = parseInt(nlayersEl.value);
      var spacing = 4;
      var startX = -((nlayers) * spacing) / 2;
      
      batchPosition = { x: startX - spacing / 2, y: -0.5, z: 0 };
      
      layerPositions = [];
      for (var i = 0; i < nlayers; i++) {
        layerPositions.push({
          x: startX + (i + 0.5) * spacing,
          y: i * 1.5,
          z: 0
        });
      }
    }
    
    function togglePause() {
      isPaused = !isPaused;
      if (isPaused) {
        pauseBtn.textContent = '▶️';
        pauseBtn.style.background = 'rgba(76, 175, 80, 0.4)';
        autoRotate = false;
      } else {
        pauseBtn.textContent = '⏸️';
        pauseBtn.style.background = 'rgba(255,255,255,0.2)';
        autoRotate = true;
        lastInteractionTime = Date.now();
      }
    }
    
    function resetView() {
      rotationX = -0.3;
      rotationY = 0.5;
      zoom = 1;
      isPaused = false;
      autoRotate = true;
      pauseBtn.textContent = '⏸️';
      pauseBtn.style.background = 'rgba(255,255,255,0.2)';
      lastInteractionTime = Date.now();
      autoLayout();
    }
    
    pauseBtn.addEventListener('click', togglePause);
    resetBtn.addEventListener('click', resetView);
    autoLayoutBtn.addEventListener('click', autoLayout);
    
    document.addEventListener('keydown', function(e) {
      if (e.key === 'p' || e.key === 'P') togglePause();
      if (e.key === 'r' || e.key === 'R') resetView();
      if (e.key === 'a' || e.key === 'A') autoLayout();
    });
    
    function findLayerAt(mouseX, mouseY, layers) {
      var minDist = Infinity;
      var foundLayer = null;
      
      var batchProj = project(batchPosition);
      var batchDist = Math.sqrt(Math.pow(batchProj.x - mouseX, 2) + Math.pow(batchProj.y - mouseY, 2));
      if (batchDist < 50) {
        minDist = batchDist;
        foundLayer = { type: 'batch', index: -1 };
      }
      
      for (var i = 0; i < layerPositions.length && i < layers.length; i++) {
        var proj = project(layerPositions[i]);
        var dist = Math.sqrt(Math.pow(proj.x - mouseX, 2) + Math.pow(proj.y - mouseY, 2));
        
        if (dist < minDist && dist < 60) {
          minDist = dist;
          foundLayer = { type: 'layer', index: i };
        }
      }
      
      return foundLayer;
    }
    
    canvas.addEventListener('mousedown', function(e) {
      var rect = canvas.getBoundingClientRect();
      var mouseX = e.clientX - rect.left;
      var mouseY = e.clientY - rect.top;
      
      var layers = getCurrentLayersInfo();
      var clickedLayer = findLayerAt(mouseX, mouseY, layers);
      
      if (clickedLayer) {
        isDraggingLayer = true;
        selectedLayer = clickedLayer;
        canvas.style.cursor = 'grabbing';
      } else {
        isDragging = true;
        canvas.style.cursor = 'grabbing';
      }
      
      autoRotate = false;
      lastX = e.clientX;
      lastY = e.clientY;
      lastInteractionTime = Date.now();
    });
    
    canvas.addEventListener('mousemove', function(e) {
      var rect = canvas.getBoundingClientRect();
      var mouseX = e.clientX - rect.left;
      var mouseY = e.clientY - rect.top;
      
      if (isDraggingLayer && selectedLayer) {
        var deltaX = (e.clientX - lastX) * 0.05;
        var deltaY = -(e.clientY - lastY) * 0.05;
        
        if (selectedLayer.type === 'batch') {
          batchPosition.x += deltaX;
          batchPosition.y += deltaY;
        } else if (selectedLayer.type === 'layer') {
          layerPositions[selectedLayer.index].x += deltaX;
          layerPositions[selectedLayer.index].y += deltaY;
        }
        
        lastX = e.clientX;
        lastY = e.clientY;
      } else if (isDragging) {
        var deltaX = e.clientX - lastX;
        var deltaY = e.clientY - lastY;
        rotationY += deltaX * 0.01;
        rotationX += deltaY * 0.01;
        rotationX = Math.max(-1.5, Math.min(1.5, rotationX));
        lastX = e.clientX;
        lastY = e.clientY;
      }
      
      if (!isDragging && !isDraggingLayer) {
        var layers = getCurrentLayersInfo();
        var hoveredLayer = findLayerAt(mouseX, mouseY, layers);
        canvas.style.cursor = hoveredLayer ? 'pointer' : 'grab';
      }
    });
    
    canvas.addEventListener('mouseup', function() {
      isDragging = false;
      isDraggingLayer = false;
      selectedLayer = null;
      canvas.style.cursor = 'grab';
      if (!isPaused) {
        setTimeout(function() { autoRotate = true; }, 3000);
      }
    });
    
    canvas.addEventListener('mouseleave', function() {
      isDragging = false;
      isDraggingLayer = false;
      selectedLayer = null;
      canvas.style.cursor = 'grab';
    });
    
    canvas.addEventListener('wheel', function(e) {
      e.preventDefault();
      zoom *= (1 + e.deltaY * 0.001);
      zoom = Math.max(0.5, Math.min(2, zoom));
      lastInteractionTime = Date.now();
      autoRotate = false;
      if (!isPaused) {
        setTimeout(function() { autoRotate = true; }, 3000);
      }
    });
    
    function rotateX(point, angle) {
      var cos = Math.cos(angle);
      var sin = Math.sin(angle);
      return { x: point.x, y: point.y * cos - point.z * sin, z: point.y * sin + point.z * cos };
    }
    
    function rotateY(point, angle) {
      var cos = Math.cos(angle);
      var sin = Math.sin(angle);
      return { x: point.x * cos - point.z * sin, y: point.y, z: point.x * sin + point.z * cos };
    }
    
    function project(point) {
      var rotated = rotateX(point, rotationX);
      rotated = rotateY(rotated, rotationY);
      var scale = zoom * 30;
      return { x: canvas.width / 2 + rotated.x * scale, y: canvas.height / 2 - rotated.y * scale, z: rotated.z };
    }
    
    function shadeColor(color, percent) {
      var num = parseInt(color.replace('#', ''), 16);
      var amt = Math.round(2.55 * percent);
      var R = (num >> 16) + amt;
      var G = (num >> 8 & 0x00FF) + amt;
      var B = (num & 0x0000FF) + amt;
      return '#' + (0x1000000 + (R < 255 ? R < 1 ? 0 : R : 255) * 0x10000 + (G < 255 ? G < 1 ? 0 : G : 255) * 0x100 + (B < 255 ? B < 1 ? 0 : B : 255)).toString(16).slice(1);
    }
    
    function draw3DBox(x, y, z, width, height, depth, color, opacity, label, shape) {
      shape = shape || 'box';
      if (shape === 'cylinder') { draw3DCylinder(x, y, z, width, height, depth, color, opacity, label); return; }
      if (shape === 'diamond') { draw3DDiamond(x, y, z, width, height, depth, color, opacity, label); return; }
      
      var vertices = [
        {x: x - width/2, y: y - height/2, z: z - depth/2},
        {x: x + width/2, y: y - height/2, z: z - depth/2},
        {x: x + width/2, y: y + height/2, z: z - depth/2},
        {x: x - width/2, y: y + height/2, z: z - depth/2},
        {x: x - width/2, y: y - height/2, z: z + depth/2},
        {x: x + width/2, y: y - height/2, z: z + depth/2},
        {x: x + width/2, y: y + height/2, z: z + depth/2},
        {x: x - width/2, y: y + height/2, z: z + depth/2}
      ];
      
      var projected = vertices.map(function(v) { return project(v); });
      
      var faces = [
        {vertices: [0, 1, 2, 3], color: shadeColor(color, -20)},
        {vertices: [4, 5, 6, 7], color: shadeColor(color, 20)},
        {vertices: [0, 1, 5, 4], color: shadeColor(color, -40)},
        {vertices: [2, 3, 7, 6], color: shadeColor(color, 40)},
        {vertices: [1, 2, 6, 5], color: shadeColor(color, -10)},
        {vertices: [0, 3, 7, 4], color: shadeColor(color, 10)}
      ];
      
      faces.sort(function(a, b) {
        var za = a.vertices.reduce(function(sum, i) { return sum + projected[i].z; }, 0) / 4;
        var zb = b.vertices.reduce(function(sum, i) { return sum + projected[i].z; }, 0) / 4;
        return za - zb;
      });
      
      faces.forEach(function(face) {
        ctx.beginPath();
        ctx.moveTo(projected[face.vertices[0]].x, projected[face.vertices[0]].y);
        for (var i = 1; i < face.vertices.length; i++) {
          ctx.lineTo(projected[face.vertices[i]].x, projected[face.vertices[i]].y);
        }
        ctx.closePath();
        ctx.fillStyle = face.color;
        ctx.globalAlpha = opacity;
        ctx.fill();
        ctx.globalAlpha = 1;
        ctx.strokeStyle = 'rgba(255, 255, 255, 0.5)';
        ctx.lineWidth = 1;
        ctx.stroke();
      });
      
      if (label) {
        var center = project({x: x, y: y + height/2 + 0.5, z: z});
        ctx.fillStyle = 'white';
        ctx.font = 'bold 10px Arial';
        ctx.textAlign = 'center';
        ctx.fillText(label, center.x, center.y);
      }
    }
    
    function draw3DCylinder(x, y, z, width, height, depth, color, opacity, label) {
      var segments = 12;
      var topVertices = [];
      var bottomVertices = [];
      
      for (var i = 0; i < segments; i++) {
        var angle = (i / segments) * Math.PI * 2;
        var cx = x + Math.cos(angle) * width / 2;
        var cz = z + Math.sin(angle) * depth / 2;
        topVertices.push({x: cx, y: y + height/2, z: cz});
        bottomVertices.push({x: cx, y: y - height/2, z: cz});
      }
      
      var projectedTop = topVertices.map(function(v) { return project(v); });
      var projectedBottom = bottomVertices.map(function(v) { return project(v); });
      
      for (var i = 0; i < segments; i++) {
        var next = (i + 1) % segments;
        ctx.beginPath();
        ctx.moveTo(projectedTop[i].x, projectedTop[i].y);
        ctx.lineTo(projectedTop[next].x, projectedTop[next].y);
        ctx.lineTo(projectedBottom[next].x, projectedBottom[next].y);
        ctx.lineTo(projectedBottom[i].x, projectedBottom[i].y);
        ctx.closePath();
        ctx.fillStyle = shadeColor(color, (i % 2 === 0) ? -10 : 10);
        ctx.globalAlpha = opacity;
        ctx.fill();
        ctx.globalAlpha = 1;
        ctx.strokeStyle = 'rgba(255, 255, 255, 0.5)';
        ctx.lineWidth = 1;
        ctx.stroke();
      }
      
      if (label) {
        var center = project({x: x, y: y + height/2 + 0.5, z: z});
        ctx.fillStyle = 'white';
        ctx.font = 'bold 10px Arial';
        ctx.textAlign = 'center';
        ctx.fillText(label, center.x, center.y);
      }
    }
    
    function draw3DDiamond(x, y, z, width, height, depth, color, opacity, label) {
      var vertices = [
        {x: x, y: y + height/2, z: z},
        {x: x + width/2, y: y, z: z},
        {x: x, y: y, z: z + depth/2},
        {x: x - width/2, y: y, z: z},
        {x: x, y: y, z: z - depth/2},
        {x: x, y: y - height/2, z: z}
      ];
      
      var projected = vertices.map(function(v) { return project(v); });
      
      var faces = [
        {vertices: [0, 1, 2], color: shadeColor(color, -20)},
        {vertices: [0, 2, 3], color: shadeColor(color, 20)},
        {vertices: [0, 3, 4], color: shadeColor(color, -10)},
        {vertices: [0, 4, 1], color: shadeColor(color, 10)},
        {vertices: [5, 1, 2], color: shadeColor(color, -30)},
        {vertices: [5, 2, 3], color: shadeColor(color, 30)},
        {vertices: [5, 3, 4], color: shadeColor(color, -20)},
        {vertices: [5, 4, 1], color: shadeColor(color, 20)}
      ];
      
      faces.sort(function(a, b) {
        var za = a.vertices.reduce(function(sum, i) { return sum + projected[i].z; }, 0) / 3;
        var zb = b.vertices.reduce(function(sum, i) { return sum + projected[i].z; }, 0) / 3;
        return za - zb;
      });
      
      faces.forEach(function(face) {
        ctx.beginPath();
        ctx.moveTo(projected[face.vertices[0]].x, projected[face.vertices[0]].y);
        for (var i = 1; i < face.vertices.length; i++) {
          ctx.lineTo(projected[face.vertices[i]].x, projected[face.vertices[i]].y);
        }
        ctx.closePath();
        ctx.fillStyle = face.color;
        ctx.globalAlpha = opacity;
        ctx.fill();
        ctx.globalAlpha = 1;
        ctx.strokeStyle = 'rgba(255, 255, 255, 0.5)';
        ctx.lineWidth = 1;
        ctx.stroke();
      });
      
      if (label) {
        var center = project({x: x, y: y + height/2 + 0.5, z: z});
        ctx.fillStyle = 'white';
        ctx.font = 'bold 10px Arial';
        ctx.textAlign = 'center';
        ctx.fillText(label, center.x, center.y);
      }
    }
    
    function drawImageBatch(x, y, z, width, height, numImages, color) {
      var imageDepth = 0.3;
      var gap = 0.1;
      var totalDepth = numImages * (imageDepth + gap);
      var startZ = z - totalDepth / 2;
      
      for (var i = 0; i < numImages; i++) {
        var imageZ = startZ + i * (imageDepth + gap);
        var alpha = 0.3 + (i / numImages) * 0.5;
        draw3DBox(x, y, imageZ, width, height, imageDepth, color, alpha, null, 'box');
      }
    }
    
    function drawConnection(x1, y1, z1, x2, y2, z2, animated) {
      var start = project({x: x1, y: y1, z: z1});
      var end = project({x: x2, y: y2, z: z2});
      var midX = (start.x + end.x) / 2;
      var midY = Math.min(start.y, end.y) - 20;
      
      if (animated && !isPaused) {
        var pulse = Math.sin(time * 0.002) * 0.5 + 0.5;
        ctx.strokeStyle = 'rgba(255, 255, 255, ' + (0.3 + pulse * 0.3) + ')';
        ctx.lineWidth = 1.5 + pulse;
      } else {
        ctx.strokeStyle = 'rgba(255, 255, 255, 0.5)';
        ctx.lineWidth = 1.5;
      }
      
      ctx.setLineDash([4, 4]);
      ctx.beginPath();
      ctx.moveTo(start.x, start.y);
      ctx.quadraticCurveTo(midX, midY, end.x, end.y);
      ctx.stroke();
      ctx.setLineDash([]);
    }
    
    function getCurrentLayersInfo() {
      var hw = parseInt(hwEl.value);
      var cin = parseInt(cinEl.value);
      var nlayers = parseInt(nlayersEl.value);
      var currentH = hw;
      var currentW = hw;
      var currentCin = cin;
      var currentFCInput = 0;
      var layersInfo = [];
      
      for (var i = 0; i < nlayers; i++) {
        var typeSelect = root.querySelector('#ep0903_type_' + i);
        var type = typeSelect ? typeSelect.value : 'CONV';
        var inputStr = '';
        var outputStr = '';
        
        if (type === 'CONV' || type === 'POOL') {
          inputStr = currentH + '×' + currentW + '×' + currentCin;
        } else if (type === 'FC') {
          // Se a camada anterior era FC, usa a saída dela
          if (i > 0 && layersInfo[i-1].type === 'FC') {
            inputStr = currentFCInput;
          } else {
            // Se veio de CONV/POOL, faz flatten
            inputStr = (currentH * currentW * currentCin);
            currentFCInput = inputStr;
          }
        }
        
        if (type === 'CONV') {
          var kh = parseInt(root.querySelector('#ep0903_kh_' + i).value);
          var kw = parseInt(root.querySelector('#ep0903_kw_' + i).value);
          var cout = parseInt(root.querySelector('#ep0903_cout_' + i).value);
          var outH = currentH - kh + 1;
          var outW = currentW - kw + 1;
          outputStr = outH + '×' + outW + '×' + cout;
          currentH = outH;
          currentW = outW;
          currentCin = cout;
          currentFCInput = currentH * currentW * currentCin;
        } else if (type === 'POOL') {
          var poolSize = parseInt(root.querySelector('#ep0903_pool_size_' + i).value);
          var poolOutH = Math.floor(currentH / poolSize);
          var poolOutW = Math.floor(currentW / poolSize);
          outputStr = poolOutH + '×' + poolOutW + '×' + currentCin;
          currentH = poolOutH;
          currentW = poolOutW;
          currentFCInput = currentH * currentW * currentCin;
        } else if (type === 'FC') {
          var fcout = parseInt(root.querySelector('#ep0903_fcout_' + i).value);
          outputStr = fcout;
          currentFCInput = fcout;
          // Após FC, não há mais dimensões espaciais
          currentH = 0;
          currentW = 0;
          currentCin = 0;
        }
        
        layersInfo.push({
          type: type,
          input: inputStr,
          output: outputStr,
          h: currentH,
          w: currentW,
          cin: currentCin
        });
      }
      
      return layersInfo;
    }
    
    function render3D(layers, batchSize) {
      ctx.clearRect(0, 0, canvas.width, canvas.height);
      if (!isPaused) time += 16;
      if (autoRotate && !isDragging && !isPaused && Date.now() - lastInteractionTime > 3000) rotationY += 0.005;
      
      // Grid
      ctx.strokeStyle = 'rgba(255, 255, 255, 0.08)';
      ctx.lineWidth = 0.5;
      for (var i = -10; i <= 10; i++) {
        var start = project({x: i * 2, y: -2, z: -10 * 2});
        var end = project({x: i * 2, y: -2, z: 10 * 2});
        ctx.beginPath(); ctx.moveTo(start.x, start.y); ctx.lineTo(end.x, end.y); ctx.stroke();
        start = project({x: -10 * 2, y: -2, z: i * 2});
        end = project({x: 10 * 2, y: -2, z: i * 2});
        ctx.beginPath(); ctx.moveTo(start.x, start.y); ctx.lineTo(end.x, end.y); ctx.stroke();
      }
      
      var colors = ['#4a90e2', '#50e3c2', '#f5a623', '#d0021b', '#8b572a', '#9013fe'];
      
      // Batch (apenas se a primeira camada for CONV ou POOL)
      if (layers.length > 0 && (layers[0].type === 'CONV' || layers[0].type === 'POOL')) {
        var inputWidth = Math.max(1, Math.min(4, layers[0].h / 8));
        var inputHeight = Math.max(1, Math.min(4, layers[0].w / 8));
        var labelPos = project({x: batchPosition.x, y: batchPosition.y + inputHeight/2 + 0.7, z: batchPosition.z});
        ctx.fillStyle = 'white';
        ctx.font = 'bold 9px Arial';
        ctx.textAlign = 'center';
        ctx.fillText('BATCH: ' + batchSize, labelPos.x, labelPos.y);
        drawImageBatch(batchPosition.x, batchPosition.y, batchPosition.z, inputWidth, inputHeight, batchSize, '#ff6b6b');
        
        // Conexão batch -> primeira camada
        drawConnection(batchPosition.x + inputWidth/2, batchPosition.y, batchPosition.z, layerPositions[0].x - Math.max(1, Math.min(4, layers[0].h / 8))/2, layerPositions[0].y, layerPositions[0].z, true);
      }
      
      // Conexões entre camadas
      for (var i = 0; i < layers.length - 1 && i < layerPositions.length - 1; i++) {
        drawConnection(layerPositions[i].x + 1, layerPositions[i].y, layerPositions[i].z, layerPositions[i + 1].x - 1, layerPositions[i + 1].y, layerPositions[i + 1].z, true);
      }
      
      // Camadas
      for (var i = 0; i < layers.length && i < layerPositions.length; i++) {
        var layer = layers[i];
        var pos = layerPositions[i];
        
        var color = colors[i % colors.length];
        var label = '';
        
        if (layer.type === 'CONV') {
          var visWidth = Math.max(1, Math.min(4, layer.h / 8));
          var visHeight = Math.max(1, Math.min(4, layer.w / 8));
          var visDepth = Math.max(0.3, Math.min(2, layer.cin / 4));
          label = layer.type + ': ' + layer.input + ' → ' + layer.output;
          draw3DBox(pos.x, pos.y, pos.z, visWidth, visHeight, visDepth, color, 0.7, label, 'box');
        } else if (layer.type === 'POOL') {
          var visWidth = Math.max(1, Math.min(4, layer.h / 8));
          var visHeight = Math.max(1, Math.min(4, layer.w / 8));
          var visDepth = Math.max(0.3, Math.min(2, layer.cin / 4));
          label = layer.type + ': ' + layer.input + ' → ' + layer.output;
          draw3DBox(pos.x, pos.y, pos.z, visWidth, visHeight, visDepth, color, 0.7, label, 'cylinder');
        } else if (layer.type === 'FC') {
          label = layer.type + ': ' + layer.input + ' → ' + layer.output;
          draw3DBox(pos.x, pos.y, pos.z, 1, 1, 1, color, 0.7, label, 'diamond');
        }
      }
      
      animationId = requestAnimationFrame(function() { render3D(layers, batchSize); });
    }
    
    function generateLayerConfig() {
      var nlayers = parseInt(nlayersEl.value);
      var html = '';
      
      for (var i = 0; i < nlayers; i++) {
        html += '<div style="background:white;border:1px solid #e0e0e0;border-radius:6px;padding:6px 8px;display:flex;gap:6px;align-items:center;flex-wrap:wrap;font-size:9px;">';
        html += '<span style="font-weight:bold;color:#666;">' + (i+1) + ':</span>';
        html += '<select id="ep0903_type_' + i + '" style="padding:2px;border:1px solid #ccc;border-radius:3px;font-size:9px;">';
        html += '<option value="CONV"' + (i < 2 ? ' selected' : '') + '>CONV</option>';
        html += '<option value="POOL">POOL</option>';
        html += '<option value="FC"' + (i >= 2 ? ' selected' : '') + '>FC</option>';
        html += '</select>';
        html += '<div id="ep0903_params_' + i + '" style="display:flex;gap:3px;flex-wrap:wrap;"></div>';
        html += '</div>';
      }
      
      layersConfigEl.innerHTML = html;
      
      for (var i = 0; i < nlayers; i++) {
        (function(index) {
          var typeSelect = root.querySelector('#ep0903_type_' + index);
          typeSelect.addEventListener('change', function() {
            updateLayerParams(index);
            render();
          });
          updateLayerParams(index);
        })(i);
      }
      
      autoLayout();
    }
    
    function updateLayerParams(index) {
      var typeSelect = root.querySelector('#ep0903_type_' + index);
      var paramsDiv = root.querySelector('#ep0903_params_' + index);
      var type = typeSelect.value;
      
      if (type === 'CONV') {
        paramsDiv.innerHTML = 
          '<input type="number" id="ep0903_kh_' + index + '" value="3" min="1" max="7" style="width:32px;padding:2px;border:1px solid #ccc;border-radius:3px;font-size:9px;" title="kernel h">' +
          '<span style="font-size:8px;">×</span>' +
          '<input type="number" id="ep0903_kw_' + index + '" value="3" min="1" max="7" style="width:32px;padding:2px;border:1px solid #ccc;border-radius:3px;font-size:9px;" title="kernel w">' +
          '<input type="number" id="ep0903_cout_' + index + '" value="' + (index === 0 ? '8' : '16') + '" min="1" max="64" style="width:36px;padding:2px;border:1px solid #ccc;border-radius:3px;font-size:9px;" title="filters">';
      } else if (type === 'POOL') {
        paramsDiv.innerHTML = 
          '<input type="number" id="ep0903_pool_size_' + index + '" value="2" min="2" max="4" style="width:32px;padding:2px;border:1px solid #ccc;border-radius:3px;font-size:9px;" title="pool size">';
      } else if (type === 'FC') {
        paramsDiv.innerHTML = 
          '<input type="number" id="ep0903_fcout_' + index + '" value="10" min="1" max="100" style="width:36px;padding:2px;border:1px solid #ccc;border-radius:3px;font-size:9px;" title="outputs">';
      }
      
      var inputs = paramsDiv.querySelectorAll('input');
      inputs.forEach(function(input) {
        input.addEventListener('input', render);
        input.addEventListener('change', render);
      });
    }
    
    function render() {
      var hw = parseInt(hwEl.value);
      var cin = parseInt(cinEl.value);
      var batchSize = parseInt(batchEl.value);
      var nlayers = parseInt(nlayersEl.value);
      var bias = biasEl.checked ? 1 : 0;
      
      hwvEl.textContent = hw + '×' + hw;
      cinvEl.textContent = cin;
      batchvEl.textContent = batchSize;
      nlayersvEl.textContent = nlayers;
      
      var currentH = hw;
      var currentW = hw;
      var currentCin = cin;
      var currentFCInput = 0;
      var totalParams = 0;
      var layersInfo = [];
      var summaryHTML = '';
      
      for (var i = 0; i < nlayers; i++) {
        var typeSelect = root.querySelector('#ep0903_type_' + i);
        var type = typeSelect.value;
        var params = 0;
        var inputStr = '';
        var outputStr = '';
        
        // Determinar entrada
        if (type === 'CONV' || type === 'POOL') {
          inputStr = currentH + '×' + currentW + '×' + currentCin;
        } else if (type === 'FC') {
          if (i > 0 && layersInfo[i-1].type === 'FC') {
            inputStr = currentFCInput;
          } else {
            inputStr = (currentH * currentW * currentCin);
            currentFCInput = inputStr;
          }
        }
        
        // Processar camada
        if (type === 'CONV') {
          var kh = parseInt(root.querySelector('#ep0903_kh_' + i).value);
          var kw = parseInt(root.querySelector('#ep0903_kw_' + i).value);
          var cout = parseInt(root.querySelector('#ep0903_cout_' + i).value);
          var outH = currentH - kh + 1;
          var outW = currentW - kw + 1;
          params = kh * kw * currentCin * cout + cout * bias;
          outputStr = outH + '×' + outW + '×' + cout;
          currentH = outH;
          currentW = outW;
          currentCin = cout;
          currentFCInput = currentH * currentW * currentCin;
        } else if (type === 'POOL') {
          var poolSize = parseInt(root.querySelector('#ep0903_pool_size_' + i).value);
          var poolOutH = Math.floor(currentH / poolSize);
          var poolOutW = Math.floor(currentW / poolSize);
          params = 0;
          outputStr = poolOutH + '×' + poolOutW + '×' + currentCin;
          currentH = poolOutH;
          currentW = poolOutW;
          currentFCInput = currentH * currentW * currentCin;
        } else if (type === 'FC') {
          var fcout = parseInt(root.querySelector('#ep0903_fcout_' + i).value);
          var fcin = parseInt(inputStr);
          params = fcin * fcout + fcout * bias;
          outputStr = fcout;
          currentFCInput = fcout;
          currentH = 0;
          currentW = 0;
          currentCin = 0;
        }
        
        totalParams += params;
        layersInfo.push({
          type: type,
          input: inputStr,
          output: outputStr,
          params: params,
          h: currentH,
          w: currentW,
          cin: currentCin
        });
        
        summaryHTML += '<span style="color:' + (type === 'CONV' ? '#2980b9' : type === 'POOL' ? '#666' : '#009933') + ';font-weight:bold;">' + (i+1) + ' (' + type + '):</span> ';
        summaryHTML += inputStr + ' → ' + outputStr;
        summaryHTML += ' [' + params.toLocaleString('pt-BR') + ']<br>';
      }
      
      summaryHTML += '<b>Total: ' + totalParams.toLocaleString('pt-BR') + ' parâmetros</b>';
      summaryEl.innerHTML = summaryHTML;
      
      if (animationId) cancelAnimationFrame(animationId);
      render3D(layersInfo, batchSize);
    }
    
    // Inicializar
    autoLayout();
    generateLayerConfig();
    render();
    
    // Event listeners
    hwEl.addEventListener('input', render);
    cinEl.addEventListener('input', render);
    batchEl.addEventListener('input', render);
    nlayersEl.addEventListener('input', function() { generateLayerConfig(); render(); });
    biasEl.addEventListener('change', render);
  }
  
  function tryInit(){
    var root = document.getElementById('sim-ep0903');
    if(root) init(root); else setTimeout(tryInit, 200);
  }
  tryInit();
})();
</script>
''')

**Figure 9.3:** Simulator EP09_03: Parameter Counting — Convolution vs. Fully Connected Layer


<figure id="fig-09-sim-ep0903">
  <img src="imagens/fig-09-sim-ep0903.png" alt=" Simulator EP09_03: Parameter Counting — Convolution vs. Fully Connected Layer " style="max-width:80%" />
  <figcaption><strong>Figure 9.3:</strong>  Simulator EP09_03: Parameter Counting — Convolution vs. Fully Connected Layer </figcaption>
</figure>

In [ ]:
%%writefile EP09_03.py
# Python code

In [ ]:
TestSuite("EP09_03.py").run()

### EP09_04 🟡 Intersection over Union (IoU) and Non-Maximum Suppression (NMS)

Object detection models can produce **multiple candidate bounding boxes** for the same object, with different positions and confidence scores. The post-processing step responsible for eliminating these redundant detections is **Non-Maximum Suppression (NMS)**, whose fundamental operation uses the **Intersection over Union (IoU)** metric.

NMS uses this measure to decide which boxes should be kept. In general, the box with the highest confidence is selected first; then, boxes that have an IoU above a given threshold with the selected box are considered redundant and removed. The process is repeated until no candidate boxes remain.

In this exercise, you must implement the NMS algorithm from scratch, calculating the IoU between boxes and successively applying the selection and suppression criteria to produce the final set of detections.

#### 📋 Implementation Guidelines

1. **Input:** Read the integer $N$ (number of candidate boxes) and the real threshold $\tau$ (IoU threshold for suppression), on the same line.

2. **Boxes:** Read $N$ lines, each with five real values:

   `x1 y1 x2 y2 score`

   where $(x_1,y_1)$ represents the top-left corner, $(x_2,y_2)$ the bottom-right corner, and `score` the confidence score.

3. **Intersection over Union:** For two boxes $A$ and $B$,

   $$
   IoU(A,B)=
   \frac{\operatorname{Area}(A\cap B)}
   {\operatorname{Area}(A\cup B)}.
   $$

   The intersection area must be computed from the overlap of the intervals in $x$ and $y$. If there is no overlap, the intersection area is zero.

4. **Greedy NMS algorithm:**

   a. Sort the boxes by `score` in descending order. In case of ties, maintain the original reading order.

   b. Select the box with the highest score among the remaining boxes and add it to the output set.

   c. Compute the IoU between the selected box and **all remaining boxes**. Suppress those boxes for which

   $$
   \text{IoU} > \tau.
   $$

   d. Repeat steps (b) and (c) until no boxes remain.

5. **Output:** For each kept box, in the order it was selected, print its original index (reading position, starting at $0$) and its `score`, formatted with 4 decimal places. At the end, print:

   `Total kept: X`

#### 📌 Computational Constraints

* **Strict suppression:** only boxes with $\text{IoU} > \tau$ are suppressed. Boxes with $\text{IoU}=\tau$ are kept.
* **Original indices:** the output refers to the position in which each box was read from the input (starting at $0$), not its position after sorting.
* **Stable sorting:** in case of equal `score` values, the original reading order must be preserved.
* **Axis-aligned rectangles:** all boxes are specified by two corners, with $x_1 < x_2$ and $y_1 < y_2$ guaranteed in the input.
* **Coordinates and scores:** real values may be positive or negative, according to the limits defined by the input, but the box dimensions are always positive.

#### 🧠 Theoretical Background

| Element                | Role in detection post-processing                                                                                                        |
| ---------------------- | ---------------------------------------------------------------------------------------------------------------------------------------- |
| IoU                    | Quantifies the spatial overlap between two boxes; $\text{IoU}=1$ for identical boxes and $\text{IoU}=0$ for boxes with no overlap       |
| Confidence-based sorting | Ensures that the box with the highest `score` is analyzed first                                                                           |
| Threshold $\tau$       | Defines the amount of overlap required for a box to be considered redundant                                                              |
| Suppression            | Removes boxes that exhibit large overlap with an already selected box                                                                    |
| Distant boxes          | Have IoU close to zero and, in general, are not suppressed by this rule                                                                  |

#### 🧩 Methods from `morph.py` that may help

* `mm.IoU(boxA, boxB)` — computes the IoU metric, but expects boxes in the format $(x,y,w,h)$, i.e., top-left corner, width, and height. The input of this exercise uses the format $(x_1,y_1,x_2,y_2)$. The conversion is straightforward:

  $$
  w=x_2-x_1,\qquad h=y_2-y_1.
  $$

  Using this function is optional. The main goal of the exercise is to correctly implement the NMS selection and suppression process.

#### 📦 Input and Output Specification (VPL)

**Input:**

* Line 1: integer $N$ and real $\tau$.
* Next $N$ lines: $x_1\ y_1\ x_2\ y_2\ \text{score}$.

**Output:**

* One line per kept box, in selection order: `index score`.
* Last line: `Total kept: X`.

In [ ]:
from IPython.display import HTML
HTML('''
<div id="sim-ep0904" style="background-color:#fef9ef;border-radius:18px;border:1px solid #ede6d8;overflow:hidden;margin-top:20px;font-family:sans-serif;">
  <div style="background:#f3efe6;padding:8px 16px;font-size:12px;color:#5e5a4a;border-bottom:1px solid #e9dfcf;display:flex;justify-content:space-between;align-items:center;">
    <span>🎮 Simulator: IoU and Non-Maximum Suppression</span>
    <span style="background:#e8e0cf;border-radius:40px;padding:2px 10px;font-weight:600;font-size:10px;">🟡 NMS</span>
  </div>
  <div style="padding:16px;background:white;">
    
    <!-- Legenda compacta -->
    <div style="background:#f8f9fa;border:1px solid #dee2e6;border-radius:8px;padding:6px 10px;margin-bottom:10px;display:flex;gap:12px;justify-content:center;font-size:10px;flex-wrap:wrap;">
      <span style="display:flex;align-items:center;gap:3px;">
        <span style="width:12px;height:12px;background:#4a90e2;border:2px solid #2c5f8a;border-radius:2px;display:inline-block;"></span>
        <b>Selected box</b>
      </span>
      <span style="display:flex;align-items:center;gap:3px;">
        <span style="width:12px;height:12px;background:#50e3c2;border:2px solid #2c8a6e;border-radius:2px;display:inline-block;"></span>
        <b>Kept box</b>
      </span>
      <span style="display:flex;align-items:center;gap:3px;">
        <span style="width:12px;height:12px;background:#ff6b6b;border:2px solid #cc4444;border-radius:2px;display:inline-block;"></span>
        <b>Suppressed box</b>
      </span>
      <span style="display:flex;align-items:center;gap:3px;">
        <span style="width:12px;height:12px;background:#f5a623;border:2px solid #b87d1a;border-radius:2px;display:inline-block;"></span>
        <b>Candidate box</b>
      </span>
    </div>
    
    <!-- Controles compactos -->
    <div style="background:#fafafa;border:1px solid #ddd;border-radius:10px;padding:12px;margin-bottom:12px;">
      <div style="display:grid;grid-template-columns:repeat(4,1fr);gap:10px;margin-bottom:8px;">
        <div>
          <div style="display:flex;justify-content:space-between;font-size:10px;margin-bottom:2px;">
            <label style="font-weight:bold;color:#2980b9;">Number of boxes</label>
            <span id="ep0904_n_v" style="font-family:monospace;font-weight:bold;color:#2980b9;font-size:9px;">5</span>
          </div>
          <input id="ep0904_n" style="width:100%;accent-color:#2980b9;height:4px;" max="8" min="2" step="1" type="range" value="5">
        </div>
        <div>
          <div style="display:flex;justify-content:space-between;font-size:10px;margin-bottom:2px;">
            <label style="font-weight:bold;color:#2980b9;">Threshold τ (IoU)</label>
            <span id="ep0904_tau_v" style="font-family:monospace;font-weight:bold;color:#2980b9;font-size:9px;">0.50</span>
          </div>
          <input id="ep0904_tau" style="width:100%;accent-color:#2980b9;height:4px;" max="1.0" min="0.1" step="0.05" type="range" value="0.5">
        </div>
        <div>
          <div style="display:flex;justify-content:space-between;font-size:10px;margin-bottom:2px;">
            <label style="font-weight:bold;color:#2980b9;">Example</label>
            <span id="ep0904_example_v" style="font-family:monospace;font-weight:bold;color:#2980b9;font-size:9px;">Default</span>
          </div>
          <select id="ep0904_example" style="width:100%;padding:2px;border:1px solid #ccc;border-radius:3px;font-size:9px;">
            <option value="padrao">Default Example</option>
            <option value="agrupado">Clustered Boxes</option>
            <option value="disperso">Scattered Boxes</option>
            <option value="aninhado">Nested Boxes</option>
          </select>
        </div>
        <div style="display:flex;align-items:center;justify-content:center;">
          <button id="ep0904_run_btn" style="background:#2980b9;color:white;border:none;padding:6px 12px;border-radius:16px;cursor:pointer;font-size:10px;font-weight:bold;transition:all 0.3s;">▶️ Run NMS</button>
        </div>
      </div>
      
      <!-- Caixas configuráveis -->
      <div id="ep0904_boxes_config" style="display:flex;flex-wrap:wrap;gap:6px;margin-bottom:8px;">
        <!-- Gerado dinamicamente -->
      </div>
    </div>
    
    <!-- Visualização -->
    <div style="display:grid;grid-template-columns:1fr 1fr;gap:12px;margin-bottom:12px;">
      <!-- Canvas da visualização -->
      <div style="position:relative;background:linear-gradient(135deg, #1e3c72 0%, #2a5298 100%);border-radius:10px;padding:16px;min-height:300px;">
        <div style="position:absolute;top:8px;left:10px;color:white;font-weight:bold;font-size:14px;text-shadow:2px 2px 4px rgba(0,0,0,0.5);z-index:10;">🎯 Box Visualization</div>
        <canvas id="ep0904_canvas" style="width:100%;height:280px;display:block;"></canvas>
      </div>
      
      <!-- Passo a passo -->
      <div style="background:#f8f9fa;border:1px solid #dee2e6;border-radius:10px;padding:12px;overflow-y:auto;max-height:310px;">
        <div style="font-weight:bold;font-size:12px;color:#333;margin-bottom:8px;">📋 NMS Step by Step</div>
        <div id="ep0904_steps" style="font-family:monospace;font-size:10px;line-height:1.6;">
          <!-- Gerado dinamicamente -->
        </div>
      </div>
    </div>
    
    <!-- Resumo -->
    <div id="ep0904_summary" style="background:#e3f2fd;border-radius:6px;padding:8px;border:1px solid #bbdefb;font-family:monospace;font-size:10px;color:#1565c0;line-height:1.5;">
    </div>
  </div>
</div>

<script>
(function(){
  function init(root){
    if(!root || root.dataset.init) return;
    root.dataset.init = "1";
    
    // Elementos DOM
    var nEl = root.querySelector('#ep0904_n'), nvEl = root.querySelector('#ep0904_n_v');
    var tauEl = root.querySelector('#ep0904_tau'), tauvEl = root.querySelector('#ep0904_tau_v');
    var exampleEl = root.querySelector('#ep0904_example');
    var boxesConfigEl = root.querySelector('#ep0904_boxes_config');
    var canvas = root.querySelector('#ep0904_canvas');
    var ctx = canvas.getContext('2d');
    var stepsEl = root.querySelector('#ep0904_steps');
    var summaryEl = root.querySelector('#ep0904_summary');
    var runBtn = root.querySelector('#ep0904_run_btn');
    
    // Estado
    var boxes = [];
    var selectedBoxes = [];
    var suppressedBoxes = [];
    
    // Exemplos pré-definidos
    var examples = {
      padrao: [
        {x1: 10, y1: 10, x2: 50, y2: 50, score: 0.9},
        {x1: 20, y1: 20, x2: 60, y2: 60, score: 0.8},
        {x1: 30, y1: 15, x2: 70, y2: 55, score: 0.7},
        {x1: 80, y1: 80, x2: 120, y2: 120, score: 0.6},
        {x1: 85, y1: 85, x2: 125, y2: 125, score: 0.5}
      ],
      agrupado: [
        {x1: 10, y1: 10, x2: 50, y2: 50, score: 0.9},
        {x1: 15, y1: 15, x2: 55, y2: 55, score: 0.85},
        {x1: 20, y1: 20, x2: 60, y2: 60, score: 0.8},
        {x1: 25, y1: 25, x2: 65, y2: 65, score: 0.75},
        {x1: 30, y1: 30, x2: 70, y2: 70, score: 0.7}
      ],
      disperso: [
        {x1: 10, y1: 10, x2: 40, y2: 40, score: 0.9},
        {x1: 80, y1: 10, x2: 110, y2: 40, score: 0.8},
        {x1: 10, y1: 80, x2: 40, y2: 110, score: 0.7},
        {x1: 80, y1: 80, x2: 110, y2: 110, score: 0.6},
        {x1: 45, y1: 45, x2: 75, y2: 75, score: 0.5}
      ],
      aninhado: [
        {x1: 10, y1: 10, x2: 90, y2: 90, score: 0.9},
        {x1: 20, y1: 20, x2: 80, y2: 80, score: 0.8},
        {x1: 30, y1: 30, x2: 70, y2: 70, score: 0.7},
        {x1: 40, y1: 40, x2: 60, y2: 60, score: 0.6},
        {x1: 45, y1: 45, x2: 55, y2: 55, score: 0.5}
      ]
    };
    
    // Ajustar tamanho do canvas
    function resizeCanvas() {
      canvas.width = canvas.clientWidth;
      canvas.height = canvas.clientHeight;
    }
    resizeCanvas();
    window.addEventListener('resize', function() {
      resizeCanvas();
      render();
    });
    
    // Carregar exemplo
    function loadExample(name) {
      boxes = JSON.parse(JSON.stringify(examples[name]));
      selectedBoxes = [];
      suppressedBoxes = [];
      generateBoxesConfig();
      render();
      stepsEl.innerHTML = 'Clique em "Executar NMS" para processar as caixas.';
      summaryEl.innerHTML = 'Caixas carregadas: ' + boxes.length + '. Clique em "Executar NMS".';
    }
    
    // Gerar configuração das caixas
    function generateBoxesConfig() {
      var html = '';
      for (var i = 0; i < boxes.length; i++) {
        html += '<div style="background:white;border:1px solid #e0e0e0;border-radius:6px;padding:4px 6px;font-size:9px;">';
        html += '<span style="font-weight:bold;color:#666;">#' + i + ':</span>';
        html += '<input type="number" id="ep0904_x1_' + i + '" value="' + boxes[i].x1 + '" style="width:30px;padding:1px;border:1px solid #ccc;border-radius:2px;font-size:8px;" title="x1">';
        html += '<input type="number" id="ep0904_y1_' + i + '" value="' + boxes[i].y1 + '" style="width:30px;padding:1px;border:1px solid #ccc;border-radius:2px;font-size:8px;" title="y1">';
        html += '<input type="number" id="ep0904_x2_' + i + '" value="' + boxes[i].x2 + '" style="width:30px;padding:1px;border:1px solid #ccc;border-radius:2px;font-size:8px;" title="x2">';
        html += '<input type="number" id="ep0904_y2_' + i + '" value="' + boxes[i].y2 + '" style="width:30px;padding:1px;border:1px solid #ccc;border-radius:2px;font-size:8px;" title="y2">';
        html += '<input type="number" id="ep0904_score_' + i + '" value="' + boxes[i].score + '" step="0.05" min="0" max="1" style="width:30px;padding:1px;border:1px solid #ccc;border-radius:2px;font-size:8px;" title="score">';
        html += '</div>';
      }
      boxesConfigEl.innerHTML = html;
      
      // Adicionar event listeners
      for (var i = 0; i < boxes.length; i++) {
        (function(index) {
          ['x1', 'y1', 'x2', 'y2', 'score'].forEach(function(field) {
            var input = root.querySelector('#ep0904_' + field + '_' + index);
            if (input) {
              input.addEventListener('input', function() {
                boxes[index][field] = parseFloat(input.value) || 0;
                selectedBoxes = [];
                suppressedBoxes = [];
                render();
              });
            }
          });
        })(i);
      }
    }
    
    // Calcular IoU
    function calculateIoU(box1, box2) {
      var x1 = Math.max(box1.x1, box2.x1);
      var y1 = Math.max(box1.y1, box2.y1);
      var x2 = Math.min(box1.x2, box2.x2);
      var y2 = Math.min(box1.y2, box2.y2);
      
      var intersectionArea = Math.max(0, x2 - x1) * Math.max(0, y2 - y1);
      if (intersectionArea === 0) return 0;
      
      var area1 = (box1.x2 - box1.x1) * (box1.y2 - box1.y1);
      var area2 = (box2.x2 - box2.x1) * (box2.y2 - box2.y1);
      var unionArea = area1 + area2 - intersectionArea;
      
      return intersectionArea / unionArea;
    }
    
    // Executar NMS
    function runNMS() {
      var tau = parseFloat(tauEl.value);
      var remaining = boxes.map(function(box, index) {
        return { box: box, originalIndex: index };
      });
      
      // Ordenar por score decrescente (estável)
      remaining.sort(function(a, b) {
        if (b.box.score !== a.box.score) {
          return b.box.score - a.box.score;
        }
        return a.originalIndex - b.originalIndex;
      });
      
      selectedBoxes = [];
      suppressedBoxes = [];
      var steps = [];
      
      while (remaining.length > 0) {
        var selected = remaining.shift();
        selectedBoxes.push(selected);
        
        steps.push({
          type: 'select',
          box: selected,
          remaining: remaining.slice()
        });
        
        var newRemaining = [];
        var suppressed = [];
        
        for (var i = 0; i < remaining.length; i++) {
          var iou = calculateIoU(selected.box, remaining[i].box);
          if (iou > tau) {
            suppressed.push({ box: remaining[i], iou: iou });
            suppressedBoxes.push(remaining[i]);
          } else {
            newRemaining.push(remaining[i]);
          }
        }
        
        if (suppressed.length > 0) {
          steps.push({
            type: 'suppress',
            box: selected,
            suppressed: suppressed,
            remaining: newRemaining.slice()
          });
        }
        
        remaining = newRemaining;
      }
      
      return steps;
    }
    
    // Renderizar visualização
    function render() {
      var tau = parseFloat(tauEl.value);
      nvEl.textContent = boxes.length;
      tauvEl.textContent = tau.toFixed(2);
      
      // Limpar canvas
      ctx.clearRect(0, 0, canvas.width, canvas.height);
      
      // Desenhar todas as caixas
      boxes.forEach(function(box, index) {
        var isSelected = selectedBoxes.some(function(s) { return s.originalIndex === index; });
        var isSuppressed = suppressedBoxes.some(function(s) { return s.originalIndex === index; });
        
        var color = '#f5a623'; // candidata
        if (isSelected) color = '#4a90e2'; // selecionada
        if (isSuppressed) color = '#ff6b6b'; // suprimida
        
        drawBox(box, color, index);
      });
      
      // Atualizar resumo
      var summaryHTML = '';
      if (selectedBoxes.length > 0) {
        summaryHTML += '<b>Caixas selecionadas (em ordem):</b><br>';
        selectedBoxes.forEach(function(s) {
          summaryHTML += '#' + s.originalIndex + ' (score: ' + s.box.score.toFixed(4) + ')<br>';
        });
        summaryHTML += '<b>Total mantidas: ' + selectedBoxes.length + '</b>';
        summaryEl.innerHTML = summaryHTML;
      }
    }
    
    // Desenhar caixa
    function drawBox(box, color, index) {
      var scale = 2.0;
      var offsetX = 30;
      var offsetY = 30;
      
      var x = offsetX + box.x1 * scale;
      var y = offsetY + box.y1 * scale;
      var w = (box.x2 - box.x1) * scale;
      var h = (box.y2 - box.y1) * scale;
      
      // Desenhar caixa
      ctx.strokeStyle = color;
      ctx.lineWidth = 3;
      ctx.strokeRect(x, y, w, h);
      
      // Preenchimento translúcido
      ctx.fillStyle = color;
      ctx.globalAlpha = 0.2;
      ctx.fillRect(x, y, w, h);
      ctx.globalAlpha = 1;
      
      // Label
      ctx.fillStyle = color;
      ctx.font = 'bold 12px Arial';
      ctx.textAlign = 'center';
      ctx.fillText('#' + index, x + w/2, y - 5);
      
      // Score
      ctx.fillStyle = '#666';
      ctx.font = '10px Arial';
      ctx.fillText('score: ' + box.score.toFixed(2), x + w/2, y + h/2);
    }
    
    // Mostrar passo a passo
    function showSteps(steps) {
      var html = '';
      
      steps.forEach(function(step, index) {
        if (step.type === 'select') {
          html += '<div style="color:#4a90e2;font-weight:bold;margin-top:4px;">';
          html += 'Passo ' + (index + 1) + ': Selecionar caixa #' + step.box.originalIndex;
          html += ' (score: ' + step.box.box.score.toFixed(4) + ')';
          html += '</div>';
        } else if (step.type === 'suppress') {
          html += '<div style="color:#ff6b6b;margin-left:10px;">';
          html += '↳ Suprimir: ';
          step.suppressed.forEach(function(s, i) {
            if (i > 0) html += ', ';
            html += '#' + s.box.originalIndex;
            html += ' (IoU: ' + s.iou.toFixed(3) + ')';
          });
          html += '</div>';
        }
      });
      
      if (selectedBoxes.length > 0) {
        html += '<div style="color:#50e3c2;font-weight:bold;margin-top:8px;">';
        html += '✓ Resultado: ' + selectedBoxes.length + ' caixa(s) mantida(s)';
        html += '</div>';
      }
      
      stepsEl.innerHTML = html;
    }
    
    // Executar NMS
    function executeNMS() {
      var steps = runNMS();
      showSteps(steps);
      render();
      
      // Animação do botão
      runBtn.textContent = '✓ Executado!';
      runBtn.style.background = '#50e3c2';
      setTimeout(function() {
        runBtn.textContent = '▶️ Executar NMS';
        runBtn.style.background = '#2980b9';
      }, 1000);
    }
    
    // Event listeners
    runBtn.addEventListener('click', executeNMS);
    
    nEl.addEventListener('input', function() {
      var n = parseInt(nEl.value);
      var currentN = boxes.length;
      
      if (n > currentN) {
        for (var i = currentN; i < n; i++) {
          boxes.push({
            x1: 10 + i * 5,
            y1: 10 + i * 5,
            x2: 50 + i * 5,
            y2: 50 + i * 5,
            score: 0.9 - i * 0.1
          });
        }
      } else if (n < currentN) {
        boxes = boxes.slice(0, n);
      }
      
      selectedBoxes = [];
      suppressedBoxes = [];
      generateBoxesConfig();
      render();
      stepsEl.innerHTML = 'Clique em "Executar NMS" para processar as caixas.';
      summaryEl.innerHTML = 'Caixas atualizadas: ' + boxes.length + '. Clique em "Executar NMS".';
    });
    
    tauEl.addEventListener('input', function() {
      tauvEl.textContent = parseFloat(tauEl.value).toFixed(2);
      selectedBoxes = [];
      suppressedBoxes = [];
      render();
      stepsEl.innerHTML = 'Clique em "Executar NMS" para processar as caixas.';
      summaryEl.innerHTML = 'Limiar atualizado. Clique em "Executar NMS".';
    });
    
    exampleEl.addEventListener('change', function() {
      loadExample(exampleEl.value);
    });
    
    // Inicializar
    loadExample('padrao');
  }
  
  function tryInit(){
    var root = document.getElementById('sim-ep0904');
    if(root) init(root); else setTimeout(tryInit, 200);
  }
  tryInit();
})();
</script>
''')

**Figure 9.4:** EP09_04 Simulator: IoU and Non-Maximum Suppression (NMS)


<figure id="fig-09-sim-ep0904">
  <img src="imagens/fig-09-sim-ep0904.png" alt=" EP09_04 Simulator: IoU and Non-Maximum Suppression (NMS) " style="max-width:80%" />
  <figcaption><strong>Figure 9.4:</strong>  EP09_04 Simulator: IoU and Non-Maximum Suppression (NMS) </figcaption>
</figure>

In [ ]:
%%writefile EP09_04.py
# Python code


In [ ]:
TestSuite("EP09_04.py").run()

### EP09_05 🟠 Segmentation Evaluation: Pixel-wise IoU and Dice

Block 2 of the "Semantic Segmentation with U-Net Architecture" section defines, in a few lines, the `iou_mascaras` function, used to measure the quality of the classic morphological baseline (smoothing + Otsu + opening) and, later, of the trained U-Net itself. Unlike the IoU from EP09_04 — computed over **bounding boxes** (rectangular regions described by four numbers) —, the segmentation IoU is computed **pixel by pixel**: each image position is compared individually between the predicted mask and the reference mask.

You have been tasked with generalizing this evaluation, implementing not only the pixel-wise IoU but also the **Dice coefficient**, another overlap metric widely used in medical segmentation (including in the `perda_dice` function, mentioned in the same block of the chapter as the basis for the loss function used to train the U-Net).

#### 📋 Implementation Guidelines

1. **Input:** Read the dimensions $H \times W$ of the masks.

2. **Predicted mask:** Read $H$ lines with $W$ integer values (0 or 1) each — for example, the output of a U-Net after thresholding at $0.5$ over the sigmoid, as in Block 4 of the chapter.

3. **Reference mask:** Read another $H$ lines with $W$ integer values (0 or 1) each — the *ground truth*.

4. **Intersection and union:** Considering each pixel as belonging to the object when its value is different from zero,
   $$
   \text{intersection} = \sum_{i,j} \mathbb{1}[P_{ij}=1 \wedge R_{ij}=1], \qquad
   \text{union} = \sum_{i,j} \mathbb{1}[P_{ij}=1 \vee R_{ij}=1].
   $$

5. **Pixel-wise IoU:**
   $$
   \text{IoU} = \frac{\text{intersection}}{\text{union}}.
   $$

6. **Dice coefficient:**
   $$
   \text{Dice} = \frac{2 \cdot \text{intersection}}{|P| + |R|},
   $$
   where $|P|$ and $|R|$ are the total number of object pixels in each mask.

7. **Convention for empty masks:** if **both** masks have no object pixels (union $= 0$ and $|P|+|R|=0$), consider the correspondence trivially perfect: $\text{IoU} = \text{Dice} = 1.0$.

8. **Output:** Two lines, `IoU: X.XXXX` and `Dice: X.XXXX`, each value with 4 decimal places.

#### 📌 Computational Constraints

* **Any non-zero value counts as an object:** treat values different from $0$ (not just $1$) as belonging to the mask, replicating the `predita > 0` check used in `iou_mascaras` in the chapter.
* **Same dimensions:** the two masks always have exactly $H \times W$ elements.
* **Empty convention:** apply the rule from item 7 **only** when both masks are completely empty; if only one is empty, the intersection is $0$ and the resulting IoU/Dice will also be $0$.

#### 🧠 Theoretical Foundation

| Element | Role in segmentation evaluation |
|---|---|
| Pixel-wise IoU | Generalizes the metric from EP09_04 to arbitrarily shaped regions — not just rectangles — by comparing predicted and reference masks position by position |
| Dice coefficient | Metric related to IoU (always $\text{Dice} \ge \text{IoU}$), more sensitive to small intersections and widely used as a loss function in segmentation (`perda_dice` function from the chapter) |
| Empty mask convention | Avoids division by zero and recognizes that "no predicted object, no real object" is, by definition, a correct match |
| Classic vs. U-Net comparison | The chapter uses exactly this type of metric to numerically justify why the U-Net surpasses the morphological baseline in low-contrast scenarios |

#### 🧩 Methods from `morph.py` that may help

* `mm.readImg(h, w, dtype='uint8')` — directly reads each binary mask $h \times w$ from standard input (the values $0/1$ fit perfectly in the standard integer type).
* The `iou_mascaras` function itself, defined in Block 2 of the U-Net section of the chapter (not part of `morph.py`, but of the chapter's code), is the direct inspiration for this exercise — it is worth re-reading those few lines before coding.
* For an optional extension (not required by this EP), `mm.connectedComponents` or `mm.label0` (seen in the context of connected component analysis) would allow labeling each nodule individually and computing the IoU **per component**, instead of over the entire mask.

#### 📦 Input and Output Specification (VPL)

**Input:**

* Line 1: Integers $H$ and $W$.
* Next $H$ lines: $W$ integer values (0 or 1) — predicted mask.
* Next $H$ lines: $W$ integer values (0 or 1) — reference mask.

**Output:**

* Line 1: `IoU: X.XXXX`.
* Line 2: `Dice: X.XXXX`.

> ### 💡 Dica
>
> ##### 💡 Illustrative Example
>
> Consider a predicted mask with a $2\times2$ square of active pixels and a reference shifted by one column, overlapping on only half of the area:
>
> ```
> Predicted       Reference
> 0 0 0 0         0 0 0 0
> 0 1 1 0         0 0 1 1
> 0 1 1 0         0 0 1 1
> 0 0 0 0         0 0 0 0
> ```
>
> Intersection $=2$ pixels, union $=6$ pixels ($4+4-2$), therefore $\text{IoU}=2/6\approx0.3333$ and $\text{Dice}=2\cdot2/(4+4)=0.5000$ — note that Dice is always equal to or greater than IoU for the same overlap.

#### 📌 Examples

| Input | Output | Observation |
|---|---|---|
| 4 4<br>0 0 0 0<br>0 1 1 0<br>0 1 1 0<br>0 0 0 0<br>0 0 0 0<br>0 0 1 1<br>0 0 1 1<br>0 0 0 0 | IoU: 0.3333<br>Dice: 0.5000 | $4\times4$ masks with partial overlap of 2 pixels. |

In [ ]:
from IPython.display import HTML
HTML('''
<div id="sim-ep0905" style="background-color:#fef9ef;border-radius:18px;border:1px solid #ede6d8;overflow:hidden;margin-top:20px;font-family:sans-serif;">
  <div style="background:#f3efe6;padding:8px 16px;font-size:12px;color:#5e5a4a;border-bottom:1px solid #e9dfcf;display:flex;justify-content:space-between;align-items:center;">
    <span>🎮 Simulator: IoU and Dice Pixel by Pixel</span>
    <span style="background:#e8e0cf;border-radius:40px;padding:2px 10px;font-weight:600;font-size:10px;">🟠 Segmentation</span>
  </div>
  <div style="padding:16px;background:white;">
    
    <!-- Legenda compacta -->
    <div style="background:#f8f9fa;border:1px solid #dee2e6;border-radius:8px;padding:6px 10px;margin-bottom:10px;display:flex;gap:12px;justify-content:center;font-size:10px;flex-wrap:wrap;">
      <span style="display:flex;align-items:center;gap:3px;">
        <span style="width:12px;height:12px;background:#4a90e2;border:2px solid #2c5f8a;border-radius:2px;display:inline-block;"></span>
        <b>Intersection</b> (TP)
      </span>
      <span style="display:flex;align-items:center;gap:3px;">
        <span style="width:12px;height:12px;background:#ff6b6b;border:2px solid #cc4444;border-radius:2px;display:inline-block;"></span>
        <b>Only predicted</b> (FP)
      </span>
      <span style="display:flex;align-items:center;gap:3px;">
        <span style="width:12px;height:12px;background:#f5a623;border:2px solid #b87d1a;border-radius:2px;display:inline-block;"></span>
        <b>Only reference</b> (FN)
      </span>
      <span style="display:flex;align-items:center;gap:3px;">
        <span style="width:12px;height:12px;background:#f0f0f0;border:2px solid #ccc;border-radius:2px;display:inline-block;"></span>
        <b>Background</b> (TN)
      </span>
    </div>
    
    <!-- Controles compactos -->
    <div style="background:#fafafa;border:1px solid #ddd;border-radius:10px;padding:12px;margin-bottom:12px;">
      <div style="display:flex;gap:10px;margin-bottom:8px;flex-wrap:wrap;">
        <div style="flex:1;min-width:120px;">
          <div style="display:flex;justify-content:space-between;font-size:10px;margin-bottom:2px;">
            <label style="font-weight:bold;color:#2980b9;">Dimensions</label>
            <span id="ep0905_dim_v" style="font-family:monospace;font-weight:bold;color:#2980b9;font-size:9px;">5×5</span>
          </div>
          <input id="ep0905_dim" style="width:100%;accent-color:#2980b9;height:4px;" max="8" min="3" step="1" type="range" value="5">
        </div>
        <div style="flex:1;min-width:120px;">
          <div style="display:flex;justify-content:space-between;font-size:10px;margin-bottom:2px;">
            <label style="font-weight:bold;color:#2980b9;">Example</label>
            <span id="ep0905_example_v" style="font-family:monospace;font-weight:bold;color:#2980b9;font-size:9px;">Square</span>
          </div>
          <select id="ep0905_example" style="width:100%;padding:4px;border:1px solid #ccc;border-radius:3px;font-size:10px;">
            <option value="quadrado">2×2 Square</option>
            <option value="deslocado">Offset</option>
            <option value="perfeito">Perfect</option>
            <option value="vazio">Empty Masks</option>
            <option value="parcial">Partial Overlap</option>
          </select>
        </div>
        <div style="display:flex;align-items:center;gap:8px;">
          <button id="ep0905_clear_btn" style="background:#666;color:white;border:none;padding:6px 12px;border-radius:16px;cursor:pointer;font-size:10px;font-weight:bold;">🗑️ Clear</button>
          <button id="ep0905_random_btn" style="background:#f5a623;color:white;border:none;padding:6px 12px;border-radius:16px;cursor:pointer;font-size:10px;font-weight:bold;">🎲 Random</button>
        </div>
      </div>
      
      <!-- Grids de máscaras -->
      <div style="display:grid;grid-template-columns:1fr 1fr;gap:10px;margin-bottom:10px;">
        <div>
          <div style="font-weight:bold;font-size:11px;color:#333;margin-bottom:4px;text-align:center;">
            🔵 Predicted Mask
          </div>
          <div id="ep0905_pred_grid" style="display:flex;justify-content:center;overflow-x:auto;"></div>
        </div>
        <div>
          <div style="font-weight:bold;font-size:11px;color:#333;margin-bottom:4px;text-align:center;">
            🟡 Reference Mask
          </div>
          <div id="ep0905_ref_grid" style="display:flex;justify-content:center;overflow-x:auto;"></div>
        </div>
      </div>
    </div>
    
    <!-- Visualização -->
    <div style="display:grid;grid-template-columns:1fr 1fr;gap:12px;margin-bottom:12px;">
      <!-- Canvas da visualização -->
      <div style="background:linear-gradient(135deg, #1e3c72 0%, #2a5298 100%);border-radius:10px;padding:16px;">
        <div style="color:white;font-weight:bold;font-size:13px;text-align:center;margin-bottom:8px;">
          🎯 Visual Comparison
        </div>
        <canvas id="ep0905_canvas" style="width:100%;height:220px;display:block;background:white;border-radius:5px;"></canvas>
      </div>
      
      <!-- Métricas -->
      <div style="background:#f8f9fa;border:1px solid #dee2e6;border-radius:10px;padding:12px;overflow-y:auto;max-height:280px;">
        <div style="font-weight:bold;font-size:12px;color:#333;margin-bottom:8px;">
          📊 Calculations and Formulas
        </div>
        <div id="ep0905_metrics" style="font-size:10px;line-height:1.8;">
          <!-- Gerado dinamicamente -->
        </div>
      </div>
    </div>
    
    <!-- Resumo final -->
    <div id="ep0905_summary" style="background:linear-gradient(135deg, #667eea 0%, #764ba2 100%);border-radius:8px;padding:12px;font-family:monospace;font-size:14px;color:white;line-height:1.8;text-align:center;">
    </div>
  </div>
</div>

<script>
(function(){
  function init(root){
    if(!root || root.dataset.init) return;
    root.dataset.init = "1";
    
    // Elementos DOM
    var dimEl = root.querySelector('#ep0905_dim');
    var dimvEl = root.querySelector('#ep0905_dim_v');
    var exampleEl = root.querySelector('#ep0905_example');
    var predGridEl = root.querySelector('#ep0905_pred_grid');
    var refGridEl = root.querySelector('#ep0905_ref_grid');
    var canvas = root.querySelector('#ep0905_canvas');
    var ctx = canvas.getContext('2d');
    var metricsEl = root.querySelector('#ep0905_metrics');
    var summaryEl = root.querySelector('#ep0905_summary');
    var clearBtn = root.querySelector('#ep0905_clear_btn');
    var randomBtn = root.querySelector('#ep0905_random_btn');
    
    // Estado
    var predMask = [];
    var refMask = [];
    var H = 5;
    var W = 5;
    
    // Exemplos pré-definidos
    var examples = {
      quadrado: {
        pred: [[0,0,0,0,0],[0,1,1,0,0],[0,1,1,0,0],[0,0,0,0,0],[0,0,0,0,0]],
        ref: [[0,0,0,0,0],[0,1,1,0,0],[0,1,1,0,0],[0,0,0,0,0],[0,0,0,0,0]]
      },
      deslocado: {
        pred: [[0,0,0,0,0],[0,1,1,0,0],[0,1,1,0,0],[0,0,0,0,0],[0,0,0,0,0]],
        ref: [[0,0,0,0,0],[0,0,1,1,0],[0,0,1,1,0],[0,0,0,0,0],[0,0,0,0,0]]
      },
      perfeito: {
        pred: [[1,1,1,1,1],[1,1,1,1,1],[1,1,1,1,1],[1,1,1,1,1],[1,1,1,1,1]],
        ref: [[1,1,1,1,1],[1,1,1,1,1],[1,1,1,1,1],[1,1,1,1,1],[1,1,1,1,1]]
      },
      vazio: {
        pred: [[0,0,0,0,0],[0,0,0,0,0],[0,0,0,0,0],[0,0,0,0,0],[0,0,0,0,0]],
        ref: [[0,0,0,0,0],[0,0,0,0,0],[0,0,0,0,0],[0,0,0,0,0],[0,0,0,0,0]]
      },
      parcial: {
        pred: [[0,0,0,0,0],[0,1,1,1,0],[0,1,1,1,0],[0,1,1,1,0],[0,0,0,0,0]],
        ref: [[0,0,0,0,0],[0,0,1,1,0],[0,0,1,1,0],[0,0,1,1,0],[0,0,0,0,0]]
      }
    };
    
    // Ajustar tamanho do canvas
    function resizeCanvas() {
      var rect = canvas.getBoundingClientRect();
      canvas.width = rect.width;
      canvas.height = rect.height;
    }
    
    // Carregar exemplo
    function loadExample(name) {
      var example = examples[name];
      predMask = example.pred.map(function(row) { return row.slice(); });
      refMask = example.ref.map(function(row) { return row.slice(); });
      H = predMask.length;
      W = predMask[0].length;
      dimEl.value = H;
      dimvEl.textContent = H + '×' + W;
      generateGrids();
      render();
    }
    
    // Gerar grids clicáveis
    function generateGrids() {
      var predHTML = '<table style="border-collapse:collapse;">';
      var refHTML = '<table style="border-collapse:collapse;">';
      
      for (var i = 0; i < H; i++) {
        predHTML += '<tr>';
        refHTML += '<tr>';
        for (var j = 0; j < W; j++) {
          predHTML += '<td data-type="pred" data-i="' + i + '" data-j="' + j + '" style="width:28px;height:28px;background:' + (predMask[i][j] ? '#4a90e2' : 'white') + ';border:1px solid #999;cursor:pointer;text-align:center;font-size:10px;color:' + (predMask[i][j] ? 'white' : '#999') + ';">' + (predMask[i][j] ? '1' : '0') + '</td>';
          refHTML += '<td data-type="ref" data-i="' + i + '" data-j="' + j + '" style="width:28px;height:28px;background:' + (refMask[i][j] ? '#f5a623' : 'white') + ';border:1px solid #999;cursor:pointer;text-align:center;font-size:10px;color:' + (refMask[i][j] ? 'white' : '#999') + ';">' + (refMask[i][j] ? '1' : '0') + '</td>';
        }
        predHTML += '</tr>';
        refHTML += '</tr>';
      }
      
      predHTML += '</table>';
      refHTML += '</table>';
      
      predGridEl.innerHTML = predHTML;
      refGridEl.innerHTML = refHTML;
      
      // Adicionar event listeners
      predGridEl.querySelectorAll('td[data-type="pred"]').forEach(function(cell) {
        cell.addEventListener('click', function() {
          var i = parseInt(cell.getAttribute('data-i'));
          var j = parseInt(cell.getAttribute('data-j'));
          predMask[i][j] = predMask[i][j] ? 0 : 1;
          generateGrids();
          render();
        });
      });
      
      refGridEl.querySelectorAll('td[data-type="ref"]').forEach(function(cell) {
        cell.addEventListener('click', function() {
          var i = parseInt(cell.getAttribute('data-i'));
          var j = parseInt(cell.getAttribute('data-j'));
          refMask[i][j] = refMask[i][j] ? 0 : 1;
          generateGrids();
          render();
        });
      });
    }
    
    // Calcular métricas
    function calculateMetrics() {
      var TP = 0, FP = 0, FN = 0, TN = 0;
      
      for (var i = 0; i < H; i++) {
        for (var j = 0; j < W; j++) {
          var p = predMask[i][j] ? 1 : 0;
          var r = refMask[i][j] ? 1 : 0;
          
          if (p && r) TP++;
          else if (p && !r) FP++;
          else if (!p && r) FN++;
          else TN++;
        }
      }
      
      var intersection = TP;
      var union = TP + FP + FN;
      var predCount = TP + FP;
      var refCount = TP + FN;
      
      var iou, dice;
      
      if (union === 0) {
        iou = 1.0;
        dice = 1.0;
      } else {
        iou = intersection / union;
        dice = (predCount + refCount === 0) ? 1.0 : (2 * intersection) / (predCount + refCount);
      }
      
      return { TP, FP, FN, TN, intersection, union, predCount, refCount, iou, dice };
    }
    
    // Renderizar visualização
    function render() {
      resizeCanvas();
      var m = calculateMetrics();
      
      // Limpar canvas
      ctx.fillStyle = 'white';
      ctx.fillRect(0, 0, canvas.width, canvas.height);
      
      var cellSize = Math.min(35, (canvas.width - 40) / W);
      var offsetX = (canvas.width - W * cellSize) / 2;
      var offsetY = (canvas.height - H * cellSize) / 2;
      
      // Desenhar grid
      for (var i = 0; i < H; i++) {
        for (var j = 0; j < W; j++) {
          var p = predMask[i][j] ? 1 : 0;
          var r = refMask[i][j] ? 1 : 0;
          
          var color = '#f0f0f0';
          if (p && r) color = '#4a90e2';
          else if (p && !r) color = '#ff6b6b';
          else if (!p && r) color = '#f5a623';
          
          var x = offsetX + j * cellSize;
          var y = offsetY + i * cellSize;
          
          ctx.fillStyle = color;
          ctx.fillRect(x, y, cellSize - 2, cellSize - 2);
          ctx.strokeStyle = '#999';
          ctx.lineWidth = 1;
          ctx.strokeRect(x, y, cellSize - 2, cellSize - 2);
        }
      }
      
      // Fórmulas e cálculos
      var html = '';
      html += '<div style="margin-bottom:6px;"><b>1. Contagem de pixels:</b></div>';
      html += '<div style="color:#4a90e2;">TP (interseção) = ' + m.TP + '</div>';
      html += '<div style="color:#ff6b6b;">FP (só predita) = ' + m.FP + '</div>';
      html += '<div style="color:#f5a623;">FN (só referência) = ' + m.FN + '</div>';
      html += '<div style="color:#999;">TN (fundo) = ' + m.TN + '</div>';
      
      html += '<div style="margin-top:8px;margin-bottom:4px;"><b>2. Interseção e União:</b></div>';
      html += '<div>Interseção = TP = <b>' + m.intersection + '</b></div>';
      html += '<div>União = TP + FP + FN = ' + m.TP + ' + ' + m.FP + ' + ' + m.FN + ' = <b>' + m.union + '</b></div>';
      html += '<div>|P| = TP + FP = ' + m.TP + ' + ' + m.FP + ' = <b>' + m.predCount + '</b></div>';
      html += '<div>|R| = TP + FN = ' + m.TP + ' + ' + m.FN + ' = <b>' + m.refCount + '</b></div>';
      
      html += '<div style="margin-top:8px;margin-bottom:4px;"><b>3. Fórmulas:</b></div>';
      
      if (m.union === 0) {
        html += '<div style="color:#888;">IoU = 1.0 (máscaras vazias)</div>';
        html += '<div style="color:#888;">Dice = 1.0 (máscaras vazias)</div>';
      } else {
        html += '<div>IoU = Interseção / União = ' + m.intersection + ' / ' + m.union + ' = <b style="color:#2980b9;">' + m.iou.toFixed(4) + '</b></div>';
        html += '<div>Dice = 2·Interseção / (|P| + |R|) = 2·' + m.intersection + ' / (' + m.predCount + ' + ' + m.refCount + ') = ' + (2 * m.intersection) + ' / ' + (m.predCount + m.refCount) + ' = <b style="color:#50e3c2;">' + m.dice.toFixed(4) + '</b></div>';
      }
      
      metricsEl.innerHTML = html;
      
      // Resumo final
      summaryEl.innerHTML = '<b>IoU: ' + m.iou.toFixed(4) + '</b> &nbsp;&nbsp;|&nbsp;&nbsp; <b>Dice: ' + m.dice.toFixed(4) + '</b>';
    }
    
    // Limpar máscaras
    function clearMasks() {
      for (var i = 0; i < H; i++) {
        for (var j = 0; j < W; j++) {
          predMask[i][j] = 0;
          refMask[i][j] = 0;
        }
      }
      generateGrids();
      render();
    }
    
    // Gerar máscaras aleatórias
    function randomMasks() {
      for (var i = 0; i < H; i++) {
        for (var j = 0; j < W; j++) {
          predMask[i][j] = Math.random() > 0.5 ? 1 : 0;
          refMask[i][j] = Math.random() > 0.5 ? 1 : 0;
        }
      }
      generateGrids();
      render();
    }
    
    // Event listeners
    clearBtn.addEventListener('click', clearMasks);
    randomBtn.addEventListener('click', randomMasks);
    
    dimEl.addEventListener('input', function() {
      H = parseInt(dimEl.value);
      W = H;
      dimvEl.textContent = H + '×' + W;
      
      var newPred = [];
      var newRef = [];
      for (var i = 0; i < H; i++) {
        newPred.push([]);
        newRef.push([]);
        for (var j = 0; j < W; j++) {
          newPred[i].push(i < predMask.length && j < predMask[0].length ? predMask[i][j] : 0);
          newRef[i].push(i < refMask.length && j < refMask[0].length ? refMask[i][j] : 0);
        }
      }
      predMask = newPred;
      refMask = newRef;
      generateGrids();
      render();
    });
    
    exampleEl.addEventListener('change', function() {
      loadExample(exampleEl.value);
    });
    
    // Inicializar
    loadExample('quadrado');
  }
  
  function tryInit(){
    var root = document.getElementById('sim-ep0905');
    if(root) init(root); else setTimeout(tryInit, 200);
  }
  tryInit();
})();
</script>
''')

**Figure 9.5:** Simulator EP09_05: Segmentation Evaluation — IoU and Dice Pixel by Pixel


<figure id="fig-09-sim-ep0905">
  <img src="imagens/fig-09-sim-ep0905.png" alt=" Simulator EP09_05: Segmentation Evaluation — IoU and Dice Pixel by Pixel " style="max-width:80%" />
  <figcaption><strong>Figure 9.5:</strong>  Simulator EP09_05: Segmentation Evaluation — IoU and Dice Pixel by Pixel </figcaption>
</figure>

In [ ]:
%%writefile EP09_05.py
# Python code


In [ ]:
TestSuite("EP09_05.py").run()

### EP09_06 🔴 Integrated Pipeline: From Detection to Real-World Measurement

This final exercise integrates the two detection exercises and the principle of **photogrammetry** presented in the section "Photogrammetry and Scale Reference" — exactly the same calculation implemented in the scale-reference measurement figure in this chapter. The scenario reproduces a realistic situation: a detector (Faster R-CNN or YOLO) generates **several overlapping candidate boxes** for the same object of interest; after filtering them by NMS, the surviving box with the highest confidence is used, together with a reference box of known real width (such as the $8.56$ cm card), to estimate the real dimensions of the detected object.

#### 📋 Implementation Guidelines

1. **Known reference:** Read the real value $L_{ref}$ (real width of the reference object, in cm) and then the four real values $x_1\ y_1\ x_2\ y_2$ of its bounding box in pixels (already known, with no need for detection).
2. **Candidates of the object to measure:** Read the integer $N$ (number of candidate boxes produced by the detector for the object of interest) and the real threshold $\tau$; then read the $N$ lines of candidate boxes, each with $x_1\ y_1\ x_2\ y_2\ \text{score}$.
3. **Step 1 — NMS:** Apply exactly the Non-Maximum Suppression algorithm from EP09_04 to the $N$ candidate boxes, using the threshold $\tau$, to eliminate redundant detections of the same object.
4. **Step 2 — Final box selection:** After NMS, the box with the highest `score` among those retained is the final detection of the object (the input guarantees that all candidate boxes correspond to a single physical object, so the first box selected by NMS is already the final result).
5. **Step 3 — Scale-reference measurement:** Calculate the ratio $\text{cm/pixel} = L_{ref} / \text{reference width in pixels}$ and apply it to both the width and height (in pixels) of the final object box, obtaining its estimated real dimensions in centimeters.
6. **Output:** First, one line per box retained after NMS (same format as EP09_04): `index score`. Then the line `Total kept: X`. Finally, the line `Object: L x A cm`, where $L$ and $A$ are the estimated width and height of the object, each with 2 decimal places.

#### 📌 Computational Constraints

* **Reuse the NMS from EP09_04** in full — same tie-breaking rule, same suppression criterion ($\text{IoU} > \tau$).
* **The reference does not undergo NMS:** its box is given directly, without competing candidates.
* **Single ratio for width and height:** just as in the photogrammetry figure in the chapter, the same cm/pixel ratio (derived from the reference width) is applied to both the width and height of the object — there is no separate vertical calibration.

#### 🧠 Theoretical Foundation

| Step | Chapter concept |
|---|---|
| Multiple candidate boxes | Raw output of a detector such as Faster R-CNN or YOLO, before post-processing |
| NMS (EP09_04) | Filters redundant detections, keeping only the most confident one for the object |
| Known scale reference | Same principle as the $8.56$ cm card used in the section "Photogrammetry and Scale Reference" |
| Pixel → centimeter conversion | Simple rule of three: $\text{cm/pixel} = L_{ref} / w_{ref\_px}$, applied to the final object box |

#### 🧩 `morph.py` Methods That May Help

* `mm.IoU(boxA, boxB)` — the same function suggested in EP09_04, here reused within the NMS step of this integrated *pipeline* (remember the format conversion: $w = x_2-x_1$, $h = y_2-y_1$).
* If you have already solved EP09_04 by encapsulating NMS in your own function, this is the ideal time to **reuse that code** — integrating modules that have already been tested individually is exactly the engineering practice this exercise aims to reinforce.

#### 📦 Input and Output Specification (VPL)

**Input:**

* Line 1: Real $L_{ref}$.
* Line 2: $x_1\ y_1\ x_2\ y_2$ of the reference box.
* Line 3: Integer $N$ and real $\tau$.
* Next $N$ lines: $x_1\ y_1\ x_2\ y_2\ \text{score}$ of the candidate boxes for the object.

**Output:**

* One line per box retained after NMS: `index score`.
* Next line: `Total kept: X`.
* Last line: `Object: L x A cm`.

#### 📌 Examples

| Input | Output | Observation |
|---|---|---|
| 8.56<br>30 200 170 288<br>3 0.5<br>250 100 470 250 0.92<br>255 105 468 245 0.88<br>600 600 650 650 0.40 | 0 0.9200<br>2 0.4000<br>Total kept: 2<br>Object: 13.45 x 9.17 cm | Box 1 is suppressed because it strongly overlaps box 0; the final detection of the object is box 0. |

In [ ]:
from IPython.display import HTML
HTML('''
<div id="sim-ep0906" style="background-color:#fef9ef;border-radius:18px;border:1px solid #ede6d8;overflow:hidden;margin-top:20px;font-family:sans-serif;">
  <div style="background:#f3efe6;padding:8px 16px;font-size:12px;color:#5e5a4a;border-bottom:1px solid #e9dfcf;display:flex;justify-content:space-between;align-items:center;">
    <span>🎮 Simulator: Integrated Pipeline — Detection to Measurement</span>
    <span style="background:#e8e0cf;border-radius:40px;padding:2px 10px;font-weight:600;font-size:10px;">🔴 Photogrammetry</span>
  </div>
  <div style="padding:16px;background:white;">
    
    <!-- Legenda compacta -->
    <div style="background:#f8f9fa;border:1px solid #dee2e6;border-radius:8px;padding:6px 10px;margin-bottom:10px;display:flex;gap:12px;justify-content:center;font-size:10px;flex-wrap:wrap;">
      <span style="display:flex;align-items:center;gap:3px;">
        <span style="width:12px;height:12px;background:#4a90e2;border:2px solid #2c5f8a;border-radius:2px;display:inline-block;"></span>
        <b>Selected box</b>
      </span>
      <span style="display:flex;align-items:center;gap:3px;">
        <span style="width:12px;height:12px;background:#ff6b6b;border:2px solid #cc4444;border-radius:2px;display:inline-block;"></span>
        <b>Suppressed box</b>
      </span>
      <span style="display:flex;align-items:center;gap:3px;">
        <span style="width:12px;height:12px;background:#50e3c2;border:2px solid #2c8a6e;border-radius:2px;display:inline-block;"></span>
        <b>Reference</b>
      </span>
      <span style="display:flex;align-items:center;gap:3px;">
        <span style="width:12px;height:12px;background:#f5a623;border:2px solid #b87d1a;border-radius:2px;display:inline-block;"></span>
        <b>Final object</b>
      </span>
    </div>
    
    <!-- Controles -->
    <div style="background:#fafafa;border:1px solid #ddd;border-radius:10px;padding:12px;margin-bottom:12px;">
      <div style="display:grid;grid-template-columns:1fr 1fr 1fr 1fr;gap:10px;margin-bottom:8px;">
        <div>
          <div style="display:flex;justify-content:space-between;font-size:10px;margin-bottom:2px;">
            <label style="font-weight:bold;color:#2980b9;">L_ref (cm)</label>
            <span id="ep0906_lref_v" style="font-family:monospace;font-weight:bold;color:#2980b9;font-size:9px;">8.56</span>
          </div>
          <input id="ep0906_lref" style="width:100%;accent-color:#2980b9;height:4px;" max="20" min="1" step="0.01" type="range" value="8.56">
        </div>
        <div>
          <div style="display:flex;justify-content:space-between;font-size:10px;margin-bottom:2px;">
            <label style="font-weight:bold;color:#2980b9;">Threshold τ</label>
            <span id="ep0906_tau_v" style="font-family:monospace;font-weight:bold;color:#2980b9;font-size:9px;">0.50</span>
          </div>
          <input id="ep0906_tau" style="width:100%;accent-color:#2980b9;height:4px;" max="1.0" min="0.1" step="0.05" type="range" value="0.5">
        </div>
        <div>
          <div style="display:flex;justify-content:space-between;font-size:10px;margin-bottom:2px;">
            <label style="font-weight:bold;color:#2980b9;">Example</label>
            <span id="ep0906_example_v" style="font-family:monospace;font-weight:bold;color:#2980b9;font-size:9px;">Default</span>
          </div>
          <select id="ep0906_example" style="width:100%;padding:4px;border:1px solid #ccc;border-radius:3px;font-size:10px;">
            <option value="padrao">Default Example</option>
            <option value="multiplos">Multiple Objects</option>
            <option value="agrupado">Grouped Boxes</option>
          </select>
        </div>
        <div style="display:flex;align-items:center;justify-content:center;">
          <button id="ep0906_run_btn" style="background:#2980b9;color:white;border:none;padding:8px 16px;border-radius:16px;cursor:pointer;font-size:11px;font-weight:bold;transition:all 0.3s;">
            ▶️ Run Pipeline
          </button>
        </div>
      </div>
      
      <!-- Caixas configuráveis -->
      <div id="ep0906_boxes_config" style="display:flex;flex-wrap:wrap;gap:6px;font-size:9px;">
        <!-- Gerado dinamicamente -->
      </div>
    </div>
    
    <!-- Visualização -->
    <div style="display:grid;grid-template-columns:3fr 2fr;gap:12px;margin-bottom:12px;">
      <!-- Canvas -->
      <div style="background:linear-gradient(135deg, #1e3c72 0%, #2a5298 100%);border-radius:10px;padding:16px;min-height:350px;">
        <div style="color:white;font-weight:bold;font-size:13px;text-align:center;margin-bottom:8px;">
          🎯 Pipeline Visualization
        </div>
        <canvas id="ep0906_canvas" style="width:100%;height:300px;display:block;background:white;border-radius:5px;"></canvas>
      </div>
      
      <!-- Passo a passo -->
      <div style="background:#f8f9fa;border:1px solid #dee2e6;border-radius:10px;padding:12px;overflow-y:auto;max-height:350px;">
        <div style="font-weight:bold;font-size:12px;color:#333;margin-bottom:8px;">
          📋 Step-by-Step Pipeline
        </div>
        <div id="ep0906_steps" style="font-size:10px;line-height:1.8;">
          <!-- Gerado dinamicamente -->
        </div>
      </div>
    </div>
    
    <!-- Resultado final -->
    <div id="ep0906_summary" style="background:linear-gradient(135deg, #667eea 0%, #764ba2 100%);border-radius:8px;padding:12px;font-family:monospace;font-size:14px;color:white;line-height:1.8;text-align:center;">
    </div>
  </div>
</div>

<script>
(function(){
  function init(root){
    if(!root || root.dataset.init) return;
    root.dataset.init = "1";
    
    // Elementos DOM
    var lrefEl = root.querySelector('#ep0906_lref');
    var lrefvEl = root.querySelector('#ep0906_lref_v');
    var tauEl = root.querySelector('#ep0906_tau');
    var tauvEl = root.querySelector('#ep0906_tau_v');
    var exampleEl = root.querySelector('#ep0906_example');
    var boxesConfigEl = root.querySelector('#ep0906_boxes_config');
    var canvas = root.querySelector('#ep0906_canvas');
    var ctx = canvas.getContext('2d');
    var stepsEl = root.querySelector('#ep0906_steps');
    var summaryEl = root.querySelector('#ep0906_summary');
    var runBtn = root.querySelector('#ep0906_run_btn');
    
    // Estado
    var refBox = { x1: 30, y1: 200, x2: 170, y2: 288 };
    var boxes = [];
    var selectedBoxes = [];
    var suppressedBoxes = [];
    var finalBox = null;
    var cmPerPixel = 0;
    
    // Exemplos
    var examples = {
      padrao: {
        refBox: { x1: 30, y1: 200, x2: 170, y2: 288 },
        boxes: [
          { x1: 250, y1: 100, x2: 470, y2: 250, score: 0.92 },
          { x1: 255, y1: 105, x2: 468, y2: 245, score: 0.88 },
          { x1: 600, y1: 600, x2: 650, y2: 650, score: 0.40 }
        ]
      },
      multiplos: {
        refBox: { x1: 20, y1: 50, x2: 100, y2: 130 },
        boxes: [
          { x1: 200, y1: 150, x2: 350, y2: 280, score: 0.85 },
          { x1: 210, y1: 160, x2: 360, y2: 290, score: 0.75 },
          { x1: 400, y1: 300, x2: 550, y2: 420, score: 0.70 },
          { x1: 410, y1: 310, x2: 560, y2: 430, score: 0.65 }
        ]
      },
      agrupado: {
        refBox: { x1: 50, y1: 50, x2: 150, y2: 150 },
        boxes: [
          { x1: 300, y1: 200, x2: 500, y2: 350, score: 0.95 },
          { x1: 310, y1: 210, x2: 490, y2: 340, score: 0.90 },
          { x1: 320, y1: 220, x2: 480, y2: 330, score: 0.85 },
          { x1: 330, y1: 230, x2: 470, y2: 320, score: 0.80 }
        ]
      }
    };
    
    // Ajustar canvas
    function resizeCanvas() {
      var rect = canvas.getBoundingClientRect();
      canvas.width = rect.width;
      canvas.height = rect.height;
    }
    
    // Carregar exemplo
    function loadExample(name) {
      var example = examples[name];
      refBox = JSON.parse(JSON.stringify(example.refBox));
      boxes = JSON.parse(JSON.stringify(example.boxes));
      selectedBoxes = [];
      suppressedBoxes = [];
      finalBox = null;
      generateBoxesConfig();
      render();
      stepsEl.innerHTML = 'Clique em "Processar Pipeline" para executar.';
      summaryEl.innerHTML = 'Aguardando processamento...';
    }
    
    // Gerar configuração das caixas
    function generateBoxesConfig() {
      var html = '';
      html += '<div style="background:#e8f5e9;border:1px solid #a5d6a7;border-radius:6px;padding:4px 6px;">';
      html += '<b>Referência:</b> ';
      html += '<input type="number" id="ep0906_ref_x1" value="' + refBox.x1 + '" style="width:35px;padding:1px;border:1px solid #ccc;border-radius:2px;font-size:8px;">';
      html += '<input type="number" id="ep0906_ref_y1" value="' + refBox.y1 + '" style="width:35px;padding:1px;border:1px solid #ccc;border-radius:2px;font-size:8px;">';
      html += '<input type="number" id="ep0906_ref_x2" value="' + refBox.x2 + '" style="width:35px;padding:1px;border:1px solid #ccc;border-radius:2px;font-size:8px;">';
      html += '<input type="number" id="ep0906_ref_y2" value="' + refBox.y2 + '" style="width:35px;padding:1px;border:1px solid #ccc;border-radius:2px;font-size:8px;">';
      html += '</div>';
      
      boxes.forEach(function(box, i) {
        html += '<div style="background:white;border:1px solid #e0e0e0;border-radius:6px;padding:4px 6px;">';
        html += '<b>#' + i + ':</b> ';
        html += '<input type="number" id="ep0906_x1_' + i + '" value="' + box.x1 + '" style="width:35px;padding:1px;border:1px solid #ccc;border-radius:2px;font-size:8px;">';
        html += '<input type="number" id="ep0906_y1_' + i + '" value="' + box.y1 + '" style="width:35px;padding:1px;border:1px solid #ccc;border-radius:2px;font-size:8px;">';
        html += '<input type="number" id="ep0906_x2_' + i + '" value="' + box.x2 + '" style="width:35px;padding:1px;border:1px solid #ccc;border-radius:2px;font-size:8px;">';
        html += '<input type="number" id="ep0906_y2_' + i + '" value="' + box.y2 + '" style="width:35px;padding:1px;border:1px solid #ccc;border-radius:2px;font-size:8px;">';
        html += '<input type="number" id="ep0906_score_' + i + '" value="' + box.score + '" step="0.05" min="0" max="1" style="width:35px;padding:1px;border:1px solid #ccc;border-radius:2px;font-size:8px;">';
        html += '</div>';
      });
      
      boxesConfigEl.innerHTML = html;
      
      // Event listeners para referência
      ['x1', 'y1', 'x2', 'y2'].forEach(function(field) {
        var input = root.querySelector('#ep0906_ref_' + field);
        if (input) {
          input.addEventListener('input', function() {
            refBox[field] = parseFloat(input.value) || 0;
            render();
          });
        }
      });
      
      // Event listeners para caixas
      boxes.forEach(function(box, i) {
        ['x1', 'y1', 'x2', 'y2', 'score'].forEach(function(field) {
          var input = root.querySelector('#ep0906_' + field + '_' + i);
          if (input) {
            input.addEventListener('input', function() {
              boxes[i][field] = parseFloat(input.value) || 0;
              selectedBoxes = [];
              suppressedBoxes = [];
              finalBox = null;
              render();
            });
          }
        });
      });
    }
    
    // Calcular IoU
    function calculateIoU(box1, box2) {
      var x1 = Math.max(box1.x1, box2.x1);
      var y1 = Math.max(box1.y1, box2.y1);
      var x2 = Math.min(box1.x2, box2.x2);
      var y2 = Math.min(box1.y2, box2.y2);
      
      var intersectionArea = Math.max(0, x2 - x1) * Math.max(0, y2 - y1);
      if (intersectionArea === 0) return 0;
      
      var area1 = (box1.x2 - box1.x1) * (box1.y2 - box1.y1);
      var area2 = (box2.x2 - box2.x1) * (box2.y2 - box2.y1);
      var unionArea = area1 + area2 - intersectionArea;
      
      return intersectionArea / unionArea;
    }
    
    // Executar pipeline
    function runPipeline() {
      var tau = parseFloat(tauEl.value);
      var lref = parseFloat(lrefEl.value);
      
      // Etapa 1: NMS
      var remaining = boxes.map(function(box, index) {
        return { box: box, originalIndex: index };
      });
      
      remaining.sort(function(a, b) {
        if (b.box.score !== a.box.score) return b.box.score - a.box.score;
        return a.originalIndex - b.originalIndex;
      });
      
      selectedBoxes = [];
      suppressedBoxes = [];
      var steps = [];
      
      steps.push('<div style="font-weight:bold;color:#333;">Etapa 1: NMS</div>');
      
      while (remaining.length > 0) {
        var selected = remaining.shift();
        selectedBoxes.push(selected);
        
        steps.push('<div style="color:#4a90e2;">Selecionar caixa #' + selected.originalIndex + ' (score: ' + selected.box.score.toFixed(4) + ')</div>');
        
        var newRemaining = [];
        for (var i = 0; i < remaining.length; i++) {
          var iou = calculateIoU(selected.box, remaining[i].box);
          if (iou > tau) {
            suppressedBoxes.push(remaining[i]);
            steps.push('<div style="color:#ff6b6b;margin-left:10px;">↳ Suprimir #' + remaining[i].originalIndex + ' (IoU: ' + iou.toFixed(3) + ')</div>');
          } else {
            newRemaining.push(remaining[i]);
          }
        }
        remaining = newRemaining;
      }
      
      steps.push('<div style="margin-top:4px;">Total mantidas: <b>' + selectedBoxes.length + '</b></div>');
      
      // Etapa 2: Seleção da caixa final
      if (selectedBoxes.length > 0) {
        finalBox = selectedBoxes[0];
        steps.push('<div style="margin-top:8px;font-weight:bold;color:#333;">Etapa 2: Caixa Final</div>');
        steps.push('<div>Caixa selecionada: #' + finalBox.originalIndex + ' (score: ' + finalBox.box.score.toFixed(4) + ')</div>');
      }
      
      // Etapa 3: Medição
      steps.push('<div style="margin-top:8px;font-weight:bold;color:#333;">Etapa 3: Medição por Referência</div>');
      
      var refWidthPx = refBox.x2 - refBox.x1;
      cmPerPixel = lref / refWidthPx;
      
      steps.push('<div>Largura da referência: ' + refWidthPx + ' pixels</div>');
      steps.push('<div>cm/pixel = ' + lref + ' / ' + refWidthPx + ' = <b>' + cmPerPixel.toFixed(6) + '</b></div>');
      
      if (finalBox) {
        var objWidthPx = finalBox.box.x2 - finalBox.box.x1;
        var objHeightPx = finalBox.box.y2 - finalBox.box.y1;
        var objWidthCm = objWidthPx * cmPerPixel;
        var objHeightCm = objHeightPx * cmPerPixel;
        
        steps.push('<div>Largura do objeto: ' + objWidthPx + ' px × ' + cmPerPixel.toFixed(6) + ' = <b>' + objWidthCm.toFixed(2) + ' cm</b></div>');
        steps.push('<div>Altura do objeto: ' + objHeightPx + ' px × ' + cmPerPixel.toFixed(6) + ' = <b>' + objHeightCm.toFixed(2) + ' cm</b></div>');
        
        stepsEl.innerHTML = steps.join('');
        
        summaryEl.innerHTML = '<b>Objeto: ' + objWidthCm.toFixed(2) + ' x ' + objHeightCm.toFixed(2) + ' cm</b>';
      } else {
        stepsEl.innerHTML = steps.join('');
        summaryEl.innerHTML = 'Nenhum objeto detectado.';
      }
      
      render();
    }
    
    // Renderizar visualização
    function render() {
      resizeCanvas();
      
      // Limpar canvas
      ctx.fillStyle = 'white';
      ctx.fillRect(0, 0, canvas.width, canvas.height);
      
      var scale = Math.min(canvas.width / 700, canvas.height / 700);
      var offsetX = 20;
      var offsetY = 20;
      
      // Desenhar caixa de referência
      drawBoxOnCanvas(refBox, '#50e3c2', 'Ref', scale, offsetX, offsetY);
      
      // Desenhar caixas
      boxes.forEach(function(box, index) {
        var isSelected = selectedBoxes.some(function(s) { return s.originalIndex === index; });
        var isSuppressed = suppressedBoxes.some(function(s) { return s.originalIndex === index; });
        var isFinal = finalBox && finalBox.originalIndex === index;
        
        var color = '#f5a623';
        var label = '#' + index;
        
        if (isFinal) {
          color = '#f5a623';
          label = '#' + index + ' ✓';
        } else if (isSelected) {
          color = '#4a90e2';
        } else if (isSuppressed) {
          color = '#ff6b6b';
          label = '#' + index + ' ✗';
        }
        
        drawBoxOnCanvas(box, color, label, scale, offsetX, offsetY);
      });
      
      // Desenhar linhas de medição
      if (finalBox && cmPerPixel > 0) {
        var refWidthPx = refBox.x2 - refBox.x1;
        var objWidthPx = finalBox.box.x2 - finalBox.box.x1;
        var objHeightPx = finalBox.box.y2 - finalBox.box.y1;
        var objWidthCm = objWidthPx * cmPerPixel;
        var objHeightCm = objHeightPx * cmPerPixel;
        
        // Linha de largura do objeto
        var y = offsetY + finalBox.box.y1 * scale - 10;
        var x1 = offsetX + finalBox.box.x1 * scale;
        var x2 = offsetX + finalBox.box.x2 * scale;
        
        ctx.strokeStyle = '#f5a623';
        ctx.lineWidth = 2;
        ctx.beginPath();
        ctx.moveTo(x1, y);
        ctx.lineTo(x2, y);
        ctx.stroke();
        
        ctx.fillStyle = '#f5a623';
        ctx.font = 'bold 10px Arial';
        ctx.textAlign = 'center';
        ctx.fillText(objWidthCm.toFixed(2) + ' cm', (x1 + x2) / 2, y - 3);
      }
    }
    
    // Desenhar caixa no canvas
    function drawBoxOnCanvas(box, color, label, scale, offsetX, offsetY) {
      var x = offsetX + box.x1 * scale;
      var y = offsetY + box.y1 * scale;
      var w = (box.x2 - box.x1) * scale;
      var h = (box.y2 - box.y1) * scale;
      
      ctx.strokeStyle = color;
      ctx.lineWidth = 3;
      ctx.strokeRect(x, y, w, h);
      
      ctx.fillStyle = color;
      ctx.globalAlpha = 0.2;
      ctx.fillRect(x, y, w, h);
      ctx.globalAlpha = 1;
      
      ctx.fillStyle = color;
      ctx.font = 'bold 11px Arial';
      ctx.textAlign = 'center';
      ctx.fillText(label, x + w/2, y - 5);
    }
    
    // Event listeners
    runBtn.addEventListener('click', function() {
      runPipeline();
      runBtn.textContent = '✓ Processado!';
      runBtn.style.background = '#50e3c2';
      setTimeout(function() {
        runBtn.textContent = '▶️ Processar Pipeline';
        runBtn.style.background = '#2980b9';
      }, 1000);
    });
    
    lrefEl.addEventListener('input', function() {
      lrefvEl.textContent = parseFloat(lrefEl.value).toFixed(2);
      selectedBoxes = [];
      suppressedBoxes = [];
      finalBox = null;
      render();
    });
    
    tauEl.addEventListener('input', function() {
      tauvEl.textContent = parseFloat(tauEl.value).toFixed(2);
      selectedBoxes = [];
      suppressedBoxes = [];
      finalBox = null;
      render();
    });
    
    exampleEl.addEventListener('change', function() {
      loadExample(exampleEl.value);
    });
    
    // Inicializar
    loadExample('padrao');
  }
  
  function tryInit(){
    var root = document.getElementById('sim-ep0906');
    if(root) init(root); else setTimeout(tryInit, 200);
  }
  tryInit();
})();
</script>
''')

**Figure 9.6:** EP09_06 Simulator: Integrated Pipeline — From Real-World Detection to Measurement


<figure id="fig-09-sim-ep0906">
  <img src="imagens/fig-09-sim-ep0906.png" alt=" EP09_06 Simulator: Integrated Pipeline — From Real-World Detection to Measurement " style="max-width:80%" />
  <figcaption><strong>Figure 9.6:</strong>  EP09_06 Simulator: Integrated Pipeline — From Real-World Detection to Measurement </figcaption>
</figure>

In [ ]:
%%writefile EP09_06.py
# Python code

In [ ]:
TestSuite("EP09_06.py").run()